# Post hoc analysis
This notebook is the main entry point for reusable post hoc analyses.
Import analysis functions from `code/src/posthoc_analysis` and run them from here.

The [5 November 2026 correctness audit](../doc/analysis_validation_2026-11-05.md) records rerun coverage, corrected mixed-ANOVA results, and remaining scope limits. Obsolete outputs in affected cells have been cleared; rerun those cells to display current results.
The [manuscript replication and theta–RT update](../doc/manuscript_replication_2026-11-05.md) records the paper comparison and the current pre-minus-post RT-improvement convention. The existing theta–RT section below uses that convention.


In [ ]:
from pathlib import Path
import sys

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
repo_root = project_root  # Repository outputs; PROJECT_ROOT below is the source-data directory.
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.config as config
importlib.reload(config)

from posthoc_analysis.config import get_subject_group, BCI_GROUP_SUBJECTS, CONTROL_GROUP_SUBJECTS
print('BCI subjects:', BCI_GROUP_SUBJECTS)
print('Control subjects:', CONTROL_GROUP_SUBJECTS)

# Training analysis file loading
Load one random training `analysis.txt` file and validate its structure.

In [ ]:
import random
from pathlib import Path
import sys

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.analysis as analysis
importlib.reload(analysis)

from posthoc_analysis.config import PROJECT_ROOT
from posthoc_analysis.analysis import load_training_analysis_file

training_analysis_files = sorted(
    PROJECT_ROOT.rglob('*_training.analysis.txt')
)
if not training_analysis_files:
    raise FileNotFoundError(
        f"No training analysis files found under PROJECT_ROOT={PROJECT_ROOT}."
    )

chosen_analysis_file = random.choice(training_analysis_files)
print('Selected analysis file:', chosen_analysis_file)

analysis_df = load_training_analysis_file(chosen_analysis_file)

print(f"Loaded {len(analysis_df)} trials")
analysis_df.head(10)

# Consolidated Trial-wise Data
Rebuild training RT from non-practice GDF Status events, validating alignment with trigger and analysis files. The canonical CSV contains actual GDF sample differences, RT in milliseconds, and response QC. This is the notebook's single RT reconstruction step.


After GDF reconstruction, explicitly pool aligned text RT <50 ms for absent GDF responses. Preserve `gdf_rt_ms` and label `rt_source = trigger_text_fallback`; this affects fast-RT reporting only.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.consolidated as consolidated
importlib.reload(consolidated)

from posthoc_analysis.consolidated import generate_consolidated_training_csv

# Rebuild GDF RTs and apply the explicit sub-50 ms text fallback
result = generate_consolidated_training_csv()

# Show a summary
print(f"CSV saved to: {result['output_path']}")
print()
print(f"Final DataFrame shape: {result['dataframe'].shape}")
if len(result['dataframe']) > 0:
    print(f"Column names: {list(result['dataframe'].columns)}")
    print()
    print("First 5 rows:")
    result['dataframe'].head()


# Comprehensive File Validation
Validate all trigger and analysis files against documentation specifications (file_contents.md, triggers.md, experiment_structure.md).

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.consolidated as consolidated
importlib.reload(consolidated)

from posthoc_analysis.consolidated import validate_all_files_comprehensive

# Run comprehensive validation
validate_all_files_comprehensive()

# Behavioral Summary Table
Create subject-level behavioral summary table for accuracy and timeout analyses.
One row per subject × group × session × trial_type with aggregated metrics.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import (
    load_and_summarize_behavioral_data,
    print_behavioral_summary_checks
)

# Load consolidated data and create behavioral summary table
behavioral_summary = load_and_summarize_behavioral_data()

print("Behavioral summary table created successfully!")
print(f"Summary table shape: {behavioral_summary['summary_table'].shape}")
print(f"Expected: 32 subjects × 2 sessions × 3 trial_types = 192 rows")
print()

# Show table structure
print("Summary table columns:")
print(list(behavioral_summary['summary_table'].columns))
print()

# Show sample data (first few rows)
print("Sample of behavioral summary table (first 12 rows):")
print(behavioral_summary['summary_table'].head(12))
print()

# Run validation
validation = behavioral_summary['validation']
print("Validation Results:")
print(f"Valid: {validation['valid']}")
if validation['issues']:
    print("Issues found:")
    for issue in validation['issues']:
        print(f"  - {issue}")
else:
    print("No validation issues found.")
print()

# Print key summary stats
summary_stats = validation['summary']
print("Key Summary Statistics:")
print(f"  Total rows: {summary_stats['total_rows']}")
print(f"  Subjects: {summary_stats['subjects']}")
print(f"  Groups: {summary_stats['groups']}")
print(f"  Sessions: {summary_stats['sessions']}")
print(f"  Trial types: {summary_stats['trial_types']}")
print()

# Run detailed sanity checks
print("Running detailed sanity checks...")
print_behavioral_summary_checks(
    behavioral_summary['summary_table'],
    behavioral_summary['original_data']
)

# Historical Analysis 2: Accuracy & Timeout

This section retains the earlier accuracy definition without RT exclusions as a sensitivity analysis. Its statistics use corrected mixed ANOVAs with participant error strata (Group × Session × Trial Type for condition-specific accuracy; Group × Session for overall accuracy and timeout). For the current distractor-task accuracy analysis excluding RT <150 ms, use the later **Distractor-task accuracy** section.


In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import analyze_accuracy_and_timeout

# Run Analysis 2
analysis2_results = analyze_accuracy_and_timeout(behavioral_summary['summary_table'])

In [ ]:
# Display figures from analysis
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("ACCURACY & TIMEOUT FIGURES")
print("="*80 + "\n")

# Figure 1: Accuracy by Trial Type
print("Figure 1: Accuracy by Trial Type (Distractor vs No Distractor)")
analysis2_results['figures']['accuracy_by_trial_type']

In [ ]:
# Figure 2: Overall Accuracy
print("\nFigure 2: Overall Accuracy")
analysis2_results['figures']['overall_accuracy']

In [ ]:
# Figure 3: Timeout Rate
print("\nFigure 3: Timeout Rate")
analysis2_results['figures']['timeout_rate']

# Analysis 1: Reaction Time
Analyze mean reaction time for correct responses during training after excluding RTs below 150 ms and removing outliers using mean ± 3 SD within each subject and session across all correct trials combined. The cleaned trials are then summarized by distractor versus no-distractor condition. This section reports warnings for partial coverage, summarizes outlier removal, runs the planned Time × Trial Type × Group analysis, and generates publication-style RT figures.


In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import analyze_reaction_time


In [ ]:
# Run Analysis 1 on the consolidated trial-level training data
analysis1_results = analyze_reaction_time(behavioral_summary['original_data'])


In [ ]:
print("\n" + "="*80)
print("REACTION TIME SUMMARY TABLES")
print("="*80 + "\n")

print("Warnings:")
for warning in analysis1_results['warnings']:
    print(f"- {warning}")

print("\nAverage percent of trials removed per subject (mean \u00b1 SEM), including pooled pre+post:")
analysis1_results['exclusion_summary_by_session']

print("\nCell-level mean RT summary:")
analysis1_results['cell_summary']


In [ ]:
print("\nTrial-removal percentages by subject \u00d7 session:")
analysis1_results['subject_session_exclusion_report']


In [ ]:
print("\nFigure 1: Reaction Time for Distractor Trials")
analysis1_results['figures']['rt_distractor']


In [ ]:
print("\nFigure 2: Reaction Time for No-Distractor Trials")
analysis1_results['figures']['rt_no_distractor']


In [ ]:
print("\nFigure 3: Reaction Time Across All Trials")
analysis1_results['figures']['rt_combined_overall']


In [ ]:
print("\nFigure 4: RT Distribution Before Outlier Removal")
analysis1_results['figures']['rt_histogram_before_outlier_removal']


## Analysis 3: Distractor cost

Compute distractor cost as mean RT on distractor trials minus mean RT on no-distractor trials after applying the same RT inclusion and outlier-removal rules used for Analysis 1.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import analyze_distractor_cost


In [ ]:
# Run Analysis 3 on the consolidated trial-level training data
analysis3_results = analyze_distractor_cost(behavioral_summary['original_data'])


In [ ]:
print("\n" + "="*80)
print("DISTRACTOR COST SUMMARY TABLES")
print("="*80 + "\n")

print("Warnings:")
for warning in analysis3_results['warnings']:
    print(f"- {warning}")

print("\nAverage percent of trials removed per subject (mean ± SEM), including pooled pre+post:")
display(analysis3_results['exclusion_summary_by_session'])

print("\nSubject-level distractor-cost table:")
display(analysis3_results['subject_session_cost'])

print("\nCell-level distractor-cost summary:")
display(analysis3_results['cell_summary'])

print("\nPlanned distractor-cost post hoc tests:")
display(analysis3_results['posthoc_tests'])


In [ ]:
print("\nFigure 5: Distractor Cost")
analysis3_results['figures']['distractor_cost']


## Stroop: Consolidate behavioral files

Create one trial-level CSV across all non-practice Stroop behavioral files, validate documented run and trial counts, and save the consolidated table for downstream Stroop analyses.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.consolidated as consolidated
importlib.reload(consolidated)

from posthoc_analysis.consolidated import generate_consolidated_stroop_csv


In [ ]:
stroop_consolidation_results = generate_consolidated_stroop_csv()


In [ ]:
print("\n" + "=" * 80)
print("STROOP CONSOLIDATION SUMMARY")
print("=" * 80)
print(f"Output path: {stroop_consolidation_results['output_path']}")
print(f"Subjects present: {len(stroop_consolidation_results['subjects_present'])}")
print(f"Runs loaded: {stroop_consolidation_results['total_runs']}")
print(f"Trials loaded: {stroop_consolidation_results['total_trials']}")
print("Collection issues:")
for key, value in stroop_consolidation_results['collection_issues'].items():
    print(f"- {key}: {len(value)}")
print("Merge issues:")
for key, value in stroop_consolidation_results['merge_issues'].items():
    print(f"- {key}: {len(value)}")


## Stroop Analysis 1: Timeout summary

Summarize Stroop timeout exclusions by run and by subject-session, then report the overall average percent of timeout trials removed across all available subject-sessions.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import load_and_analyze_stroop_timeout_data


In [ ]:
stroop_analysis1_results = load_and_analyze_stroop_timeout_data()


In [ ]:
print("\n" + "=" * 80)
print("STROOP ANALYSIS 1 SUMMARY")
print("=" * 80)
print("Warnings:")
for warning in stroop_analysis1_results['warnings']:
    print(f"- {warning}")

print("\nSubject-session timeout summary (averaged across runs):")
stroop_analysis1_results['subject_session_summary']

print("\nOverall timeout summary:")
stroop_analysis1_results['overall_summary']


## Stroop Analysis 2: Accuracy

Compute Stroop accuracy after removing timeout trials, summarize congruent, incongruent, and overall accuracy at the subject-session level, generate training-style figures, and run the planned pre/post by group accuracy analyses when the local stats stack is available.

The separate OLS statistics in the saved outputs below are historical and superseded by the appended **Stroop accuracy: 2 × 2 × 2 mixed ANOVA** section. Rerunning this section now uses the joint mixed ANOVA; accuracy values and figures retain their original calculation.


In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import load_and_analyze_stroop_accuracy_data


In [ ]:
stroop_analysis2_results = load_and_analyze_stroop_accuracy_data()


In [ ]:
print("\n" + "=" * 80)
print("STROOP ANALYSIS 2 SUMMARY")
print("=" * 80)
print("Warnings:")
for warning in stroop_analysis2_results['warnings']:
    print(f"- {warning}")

print("\nValidation:")
stroop_analysis2_results['validation']

print("\nSummary table:")
stroop_analysis2_results['summary_table']

print("\nStatistics:")
stroop_analysis2_results['stats']


## Stroop Analyses 3-4: Reaction time and Stroop effect

Run the Stroop RT and Stroop-effect analyses together using the same cleaned correct-trial dataset. This section reports incorrect-trial and exclusion summaries once, then shows the RT summaries and Stroop-effect summaries generated from the shared preprocessing workflow.

Saved outputs in this earlier section used the historical per-run RT filter. Rerun to update statistics/figures; the appended **Stroop RT: pooled-session ±3 SD exclusions** section contains the current validated exclusion counts and RT summaries.

The current Group × Session × Congruency mixed-ANOVA statistics are in the appended **Stroop RT: 2 × 2 × 2 mixed ANOVA** section; earlier saved separate RT OLS tests are superseded.


In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)

from posthoc_analysis.behavioral import (
    analyze_stroop_effect,
    load_and_analyze_stroop_reaction_time_data,
)


In [ ]:
stroop_analysis3_results = load_and_analyze_stroop_reaction_time_data()
stroop_analysis4_results = analyze_stroop_effect(rt_results=stroop_analysis3_results)


In [ ]:
print("\n" + "=" * 80)
print("STROOP ANALYSES 3-4 SUMMARY")
print("=" * 80)
print("Shared preprocessing warnings:")
for warning in stroop_analysis3_results['warnings']:
    print(f"- {warning}")

print("\nIncorrect-trial summary:")
display(stroop_analysis3_results['incorrect_trial_summary'])

print("\nSubject exclusion summary:")
display(stroop_analysis3_results['subject_exclusion_summary'])

print("\nOverall exclusion summary:")
display(stroop_analysis3_results['overall_exclusion_summary'])

print("\nStroop RT subject-session summary:")
display(stroop_analysis3_results['subject_session_summary'])

print("\nSubject-session Stroop effect summary:")
display(stroop_analysis4_results['subject_session_stroop_effect'])

print("\nCell summary:")
display(stroop_analysis4_results['cell_summary'])

print("\nStroop RT statistics:")
display(stroop_analysis3_results['stats'])

print("\nStroop effect statistics:")
display(stroop_analysis4_results['stats_anova'])
display(stroop_analysis4_results['posthoc_tests'])


## BCI Trial-Level Consolidation

Create the trial-level BCI CSV by combining threshold-log run values with online posterior trial values, excluding practice runs when present and validating run/trial alignment.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import generate_consolidated_bci_csv


In [ ]:
bci_consolidation_results = generate_consolidated_bci_csv()


In [ ]:
print("\n" + "=" * 80)
print("BCI CONSOLIDATION SUMMARY")
print("=" * 80)
print(f"Output path: {bci_consolidation_results['output_path']}")
print(f"Trials: {bci_consolidation_results['total_trials']}")
print(f"Runs: {bci_consolidation_results['total_runs']}")
print(f"Subjects: {len(bci_consolidation_results['subjects_present'])}")
print(f"Validation: {bci_consolidation_results['validation']}")

print("\nIssue counts:")
for issue_name, issue_values in bci_consolidation_results['issues'].items():
    print(f"- {issue_name}: {len(issue_values)}")

bci_consolidation_results['dataframe'].head()


## BCI AUC Across Sessions

Compute subject-level AUC for each session using posterior probabilities as distractor-class scores and `decoding_task` as the ground-truth label, then plot group means with SEM across Sessions 1-5.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import load_and_plot_bci_auc


In [ ]:
bci_auc_results = load_and_plot_bci_auc(variability='sem', save=True)


In [ ]:
print("\n" + "=" * 80)
print("BCI AUC SUMMARY")
print("=" * 80)
print(f"Figure path: {bci_auc_results['figure_path']}")

print("\nGroup/session summary:")
display(bci_auc_results['group_summary'])

print("\nRows excluded because labels were unavailable:")
display(bci_auc_results['missing_label_summary'])

print("\nSubject/session AUC table:")
display(bci_auc_results['subject_session_auc'])


## BCI AUC Mixed-Design ANOVA

Run a 2 x 5 mixed-design ANOVA on subject-level AUC values with group as the between-subject factor and session as the within-subject factor.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import run_bci_auc_mixed_anova


In [ ]:
bci_auc_anova_results = run_bci_auc_mixed_anova(
    bci_auc_results['subject_session_auc']
)


In [ ]:
print("\n" + "=" * 80)
print("BCI AUC MIXED-DESIGN ANOVA SUMMARY")
print("=" * 80)
display(bci_auc_anova_results['anova_table'])
bci_auc_anova_results['diagnostics']


## BCI Posterior Distributions

Plot subject-averaged posterior-probability distributions for Sessions 1 and 5, separately for BCI and mental-rehearsal groups and for distractor versus no-distractor trials.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import load_and_plot_bci_posterior_distributions


In [ ]:
bci_posterior_distribution_results = load_and_plot_bci_posterior_distributions(
    sessions=(1, 5),
    n_bins=20,
    normalize='density',
    save=True,
)


In [ ]:
print("\n" + "=" * 80)
print("BCI POSTERIOR DISTRIBUTION SUMMARY")
print("=" * 80)
print(f"Figure path: {bci_posterior_distribution_results['figure_path']}")

print("\nSubjects per group/session:")
display(bci_posterior_distribution_results['subject_counts'])

print("\nSubjects per group/session/class distribution:")
display(bci_posterior_distribution_results['subject_class_counts'])

print("\nPanel completeness checks:")
display(bci_posterior_distribution_results['panel_checks'])


## BCI Threshold Change From Session 1 to Session 5

Compute subject-level threshold changes by averaging run-level `thrR`, `thrL`, and transformed `thrN` within Sessions 1 and 5, then plotting Session 5 minus Session 1 by group.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import load_and_plot_bci_threshold_change_session1_to_session5


In [ ]:
bci_threshold_change_results = load_and_plot_bci_threshold_change_session1_to_session5(
    save=True,
)


In [ ]:
print("\n" + "=" * 80)
print("BCI THRESHOLD CHANGE SUMMARY")
print("=" * 80)
print(f"Figure path: {bci_threshold_change_results['figure_path']}")

print("\nSummary table:")
display(bci_threshold_change_results['summary'])

print("\nRun rows averaged per subject/session:")
display(bci_threshold_change_results['run_counts'])

print("\nMissing threshold values:")
display(bci_threshold_change_results['missing_thresholds'])


## BCI Threshold Change With Distractor Sides Combined

Repeat the Session 5 minus Session 1 threshold-change plot after combining left and right distractor thresholds at the subject/session level.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import load_and_plot_bci_combined_threshold_change_session1_to_session5


In [ ]:
bci_combined_threshold_change_results = load_and_plot_bci_combined_threshold_change_session1_to_session5(
    save=True,
)


In [ ]:
print("\n" + "=" * 80)
print("BCI COMBINED THRESHOLD CHANGE SUMMARY")
print("=" * 80)
print(f"Figure path: {bci_combined_threshold_change_results['figure_path']}")

print("\nCombined summary table:")
display(bci_combined_threshold_change_results['combined_summary'])

print("\nRun rows averaged per subject/session:")
display(bci_combined_threshold_change_results['run_counts'])


## BCI Combined Threshold Change Mixed-Design ANOVA

Run a 2-way mixed-design ANOVA on Session 5 minus Session 1 threshold changes, with threshold type as the within-subject factor and group as the between-subject factor.

In [ ]:
import sys
from pathlib import Path

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
sys.path.insert(0, str(code_src))

import importlib
import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import run_bci_combined_threshold_change_mixed_anova


In [ ]:
bci_combined_threshold_anova_results = run_bci_combined_threshold_change_mixed_anova(
    bci_combined_threshold_change_results['combined_subject_threshold_change']
)


In [ ]:
print("\n" + "=" * 80)
print("BCI COMBINED THRESHOLD CHANGE MIXED ANOVA SUMMARY")
print("=" * 80)

print("\nCell summary:")
display(bci_combined_threshold_anova_results['cell_summary'])

print("\nANOVA table:")
display(bci_combined_threshold_anova_results['anova_table'])

bci_combined_threshold_anova_results['diagnostics']


## Single-Run EEG Quick Look

Load one raw GDF recording, validate the expected single-run EEG structure, and plot all EEG electrodes for a 10-second segment beginning 30 seconds into the run.

In [ ]:
import sys
from pathlib import Path
import importlib

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

import posthoc_analysis.eeg as eeg
importlib.reload(eeg)

from posthoc_analysis.eeg import load_and_plot_eeg_run_segment


In [ ]:
e21_training_run1_eeg = load_and_plot_eeg_run_segment(
    subject_id='e21',
    session=1,
    run=1,
    task='training',
    start_sec=30.0,
    duration_sec=10.0,
)

e21_training_run1_eeg['fig']


## Single-Run EEG Bandpass Filtering

Apply the documented zero-phase 0.1-20 Hz bandpass filter to the same single-run analysis EEG data, then inspect the filtered time series and raw-vs-filtered PSD.

In [ ]:
import sys
from pathlib import Path
import importlib

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

import posthoc_analysis.eeg as eeg
importlib.reload(eeg)

from posthoc_analysis.eeg import load_filter_and_plot_eeg_run_segment


In [ ]:
e21_training_run1_filtered_eeg = load_filter_and_plot_eeg_run_segment(
    subject_id='e21',
    session=1,
    run=1,
    task='training',
    l_freq=0.1,
    h_freq=20.0,
    start_sec=30.0,
    duration_sec=10.0,
)

display(e21_training_run1_filtered_eeg['trace_fig'])
display(e21_training_run1_filtered_eeg['psd_fig'])


## Single-Run Stimulus-Locked EEG Epoching

Epoch the filtered analysis EEG data from -0.5 to 1.0 seconds around training stimulus-presentation triggers (8, 32, 44) read from the GDF Status channel. The returned epoch matrix is channels x samples x trials.

In [ ]:
import sys
from pathlib import Path
import importlib

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

import posthoc_analysis.eeg as eeg
importlib.reload(eeg)

from posthoc_analysis.eeg import load_filter_epoch_and_plot_training_stimulus_epochs


In [ ]:
e21_training_run1_epochs = load_filter_epoch_and_plot_training_stimulus_epochs(
    subject_id='e21',
    session=1,
    run=1,
    tmin=-0.5,
    tmax=1.0,
    plot_channel='PO7',
)

print('Epoch matrix shape (channels x samples x trials):', e21_training_run1_epochs['epoch_data_channels_samples_trials'].shape)
display(e21_training_run1_epochs['epoch_fig'])


## Single-Run Baseline Correction

Baseline-correct the filtered stimulus-locked epochs using the -0.2 to 0 second pre-stimulus interval. The baseline mean is computed separately for each channel x trial and subtracted from every timepoint in that channel x trial epoch.

In [ ]:
import sys
from pathlib import Path
import importlib

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

import posthoc_analysis.eeg as eeg
importlib.reload(eeg)

from posthoc_analysis.eeg import load_filter_epoch_baseline_correct_and_plot_training


In [ ]:
e21_training_run1_baseline = load_filter_epoch_baseline_correct_and_plot_training(
    subject_id='e21',
    session=1,
    run=1,
    tmin=-0.5,
    tmax=1.0,
    baseline_tmin=-0.2,
    baseline_tmax=0.0,
    plot_channel='PO7',
)

print('Baseline-corrected epoch shape (channels x samples x trials):', e21_training_run1_baseline['baseline_corrected_epochs'].shape)
print('Baseline sample indices:', e21_training_run1_baseline['baseline_indices'])
display(e21_training_run1_baseline['baseline_fig'])


## Distractor-Trial Reaction Time Summary

Compute subject-level mean reaction time for distractor trials only from the consolidated training CSV, then average across subjects by group and Session 1 vs Session 5.


In [ ]:
from pathlib import Path
import sys

repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in repo_candidates if (path / 'code' / 'src' / 'posthoc_analysis').exists()),
    Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis'),
)
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

from posthoc_analysis.behavioral import summarize_distractor_trial_reaction_time_from_training_csv


In [ ]:
distractor_rt_results = summarize_distractor_trial_reaction_time_from_training_csv()

print('Saved figure:', distractor_rt_results['figure_path'])
display(distractor_rt_results['cell_summary'])


## BCI Threshold Trajectories Across Runs

Plot group-averaged threshold values across BCI runs with session breaks. `thrR` and `thrL` use raw threshold-log values, and the no-distractor threshold is transformed as `1 - thrN` before averaging.


In [ ]:
from pathlib import Path
import sys

repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in repo_candidates if (path / 'code' / 'src' / 'posthoc_analysis').exists()),
    Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis'),
)
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

from posthoc_analysis.bci import load_and_plot_bci_threshold_trajectory_panels


In [ ]:
bci_threshold_trajectory_panel_results = load_and_plot_bci_threshold_trajectory_panels(
    variability='sem',
    save=True,
)

print('Distractor class threshold figure:', bci_threshold_trajectory_panel_results['distractor_figure_path'])
print('No distractor class threshold figure:', bci_threshold_trajectory_panel_results['no_distractor_figure_path'])


In [ ]:
display(bci_threshold_trajectory_panel_results['distractor_summary'])
display(bci_threshold_trajectory_panel_results['no_distractor_summary'])
display(bci_threshold_trajectory_panel_results['observed_run_counts'])


## BCI Threshold Trajectory Regression Statistics

Test whether threshold trajectories increase across the full run sequence and within each session. Distractor thresholds combine `thrR` and `thrL`; no-distractor thresholds use `1 - thrN`.


In [ ]:
from posthoc_analysis.bci import load_and_run_bci_threshold_trajectory_regression_stats


In [ ]:
bci_threshold_regression_results = load_and_run_bci_threshold_trajectory_regression_stats(
    correction='holm',
    save=True,
)

print('Saved statistics tables:')
for table_name, table_path in bci_threshold_regression_results['output_paths'].items():
    print(f'{table_name}: {table_path}')


In [ ]:
display(bci_threshold_regression_results['overall_tests'])
display(bci_threshold_regression_results['session_tests'])
display(bci_threshold_regression_results['session_slopes'])


## BCI Posterior Probability Trajectories Across Runs

Average posterior probabilities within each subject/session/run and ground-truth trial type. The plotted line is the average of subject run means across subjects, and the shaded band is the between-subject SEM of those subject run means.


In [ ]:
from posthoc_analysis.bci import load_and_plot_bci_posterior_trajectories


In [ ]:
bci_posterior_trajectory_results = load_and_plot_bci_posterior_trajectories(
    variability='sem',
    save=True,
)

print('Distractor class posterior figure:', bci_posterior_trajectory_results['distractor_figure_path'])
print('No distractor class posterior figure:', bci_posterior_trajectory_results['no_distractor_figure_path'])


In [ ]:
display(bci_posterior_trajectory_results['summary'])
display(bci_posterior_trajectory_results['trial_type_counts'])
display(bci_posterior_trajectory_results['missing_label_summary'])


## BCI Balanced Accuracy Across Sessions

Compute balanced accuracy by subject and session after excluding ambivalent BCI outputs (`decoding_bci_output == 3`). Ground-truth trial type comes from `decoding_task`; `decoding_bci_output == 1` is the predicted Pd/distractor class and `decoding_bci_output == 0` is the predicted no-Pd/no-distractor class.


In [ ]:
from posthoc_analysis.bci import load_and_plot_bci_balanced_accuracy


In [ ]:
bci_balanced_accuracy_results = load_and_plot_bci_balanced_accuracy(
    variability='sem',
    save=True,
)

print('Saved balanced accuracy figure:', bci_balanced_accuracy_results['figure_path'])


In [ ]:
display(bci_balanced_accuracy_results['group_summary'])
display(bci_balanced_accuracy_results['subject_session_balanced_accuracy'])
display(bci_balanced_accuracy_results['excluded_ambivalent'])


## Adaptive BCI Control Score Across Sessions

Compute run-level adaptive BCI control scores from threshold difficulty and chance-corrected balanced accuracy, then summarize and plot subject-session trajectories by group.


In [ ]:
from posthoc_analysis.bci import load_compute_and_plot_adaptive_bci_control_score


In [ ]:
adaptive_bci_control_results = load_compute_and_plot_adaptive_bci_control_score(
    save=True,
)

adaptive_bci_control_results["figure_paths"]


In [ ]:
display(adaptive_bci_control_results["group_summary"])
display(adaptive_bci_control_results["subject_session_scores"])


## Offline DecoderR/L Performance by Group

Load saved `decoderR` and `decoderL` `.mat` files, extract `performance.tpr` and `performance.tnr`, average the two decoder sides within each subject, and plot TPR, TNR, and balanced offline accuracy separately for the BCI and mental-rehearsal groups with subject-level dots.


In [ ]:
from posthoc_analysis.decoder import load_compute_and_plot_decoder_rl_performance


In [ ]:
decoder_rl_performance_results = load_compute_and_plot_decoder_rl_performance(
    variability='sem',
    save=True,
)

print('Saved offline decoder performance figure:', decoder_rl_performance_results['figure_path'])


In [ ]:
display(decoder_rl_performance_results['group_summary'])
display(decoder_rl_performance_results['subject_average'])
display(decoder_rl_performance_results['decoder_performance'])
display(decoder_rl_performance_results['issues'])


## BCI Separated Threshold Change Mixed-Design ANOVA

Run a 2 x 3 mixed-design ANOVA on Session 5 minus Session 1 threshold changes, keeping right-distractor (`thrR`), left-distractor (`thrL`), and transformed no-distractor (`1 - thrN`) thresholds as separate within-subject levels.


In [ ]:
import sys
from pathlib import Path
import importlib

repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in repo_candidates if (path / 'code' / 'src' / 'posthoc_analysis').exists()),
    Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis'),
)
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

import posthoc_analysis.bci as bci
importlib.reload(bci)

from posthoc_analysis.bci import run_bci_separated_threshold_change_mixed_anova



In [ ]:
if 'bci_threshold_change_results' not in globals():
    from posthoc_analysis.bci import load_and_plot_bci_threshold_change_session1_to_session5
    bci_threshold_change_results = load_and_plot_bci_threshold_change_session1_to_session5(
        save=True
    )

bci_separated_threshold_anova_results = run_bci_separated_threshold_change_mixed_anova(
    bci_threshold_change_results['subject_threshold_change']
)



In [ ]:
print("BCI SEPARATED THRESHOLD CHANGE MIXED ANOVA SUMMARY")
print("Dependent variable: Session 5 - Session 1 threshold change")
print("Threshold levels: thrR, thrL, transformed thrN")

print("\nCell summary:")
display(bci_separated_threshold_anova_results['cell_summary'])

print("\nSphericity diagnostics:")
display(bci_separated_threshold_anova_results['sphericity'])

print("\nANOVA table:")
display(bci_separated_threshold_anova_results['anova_table'])



## Direct BCI versus mental-rehearsal threshold-slope comparison

Directly test whether the full-training threshold trajectories differ between groups. Each participant contributes one slope across the complete 36-run sequence for the combined distractor threshold, mean(thrR, thrL), and one for the no-distractor difficulty threshold, 1 − thrN. BCI minus mental-rehearsal slope differences are evaluated using two-sided Welch tests; Holm correction is applied across these two threshold-class comparisons.

In [ ]:
from posthoc_analysis.bci import load_and_run_bci_threshold_trajectory_regression_stats


In [ ]:
bci_threshold_direct_slope_results = (
    load_and_run_bci_threshold_trajectory_regression_stats(
        correction='holm',
        save=True,
        output_dir=project_root / 'analyses',
    )
)


In [ ]:
display(bci_threshold_direct_slope_results['group_slope_comparisons'])
bci_threshold_direct_slope_results['output_paths']['group_slope_comparisons']


## New Session 5 training-derived lateralized feature analysis

Build new, participant-specific right-distractor-versus-no-distractor and left-distractor-versus-no-distractor feature spaces from Session 5 training-task EEG. This analysis will estimate new xDAWN filters (two components) and select the 30 highest-r² features from 200 ms through the final available epoch sample. It will not reuse the saved decoderR/decoderL transforms, normalization, or `keepIdx` feature sets.

The resulting Session 5–derived feature coordinates will then be held fixed and evaluated in labeled decoding runs from Sessions 1–5. Feature r² trajectories will be summarized separately by group and decoder side. The trial-balancing and temporal-downsampling procedures will be implemented from the user-supplied reference code before any model fitting is performed.

### Step 1 — Validate Session 5 training-model inputs

Create a manifest for every non-practice Session 5 training run before applying any filtering or fitting a model. This checks the required four runs per participant, validated trigger and analysis files, condition labels, and one-to-one GDF Status-event alignment. It also creates the two trial tables that will later be used for right-distractor-versus-no-distractor and left-distractor-versus-no-distractor model construction.

In [ ]:
from posthoc_analysis.decoder_training import build_session5_training_model_manifest


In [ ]:
session5_model_inputs = build_session5_training_model_manifest(
    project_root=PROJECT_ROOT,
)
session5_run_manifest = session5_model_inputs['manifest']
session5_training_trials = session5_model_inputs['trials']
session5_right_model_trials = session5_model_inputs['right_trials']
session5_left_model_trials = session5_model_inputs['left_trials']


In [ ]:
display(session5_model_inputs['group_summary'])
display(
    session5_run_manifest[
        [
            'group', 'subject_id', 'run_id', 'n_trials',
            'n_no_distractor', 'n_distractor_right', 'n_distractor_left',
            'n_right_model_trials', 'n_left_model_trials',
            'status_to_trigger_time_slope',
            'max_trigger_time_residual_samples', 'status_event_alignment',
        ]
    ]
)


### Step 2 — Zero-phase preprocessing and Status-anchored epochs

Verify the 61-label scalp EEG layout across the validated Session 5 runs, then preprocess one participant at a time to keep intermediate arrays inspectable. Each run is filtered with a zero-phase 0.1–20 Hz FIR bandpass, epoched from −0.5 to +1.0 s around the GDF Status stimulus samples, and baseline-corrected per channel and trial from −0.2 to 0 s. No lateralization, xDAWN, balancing, pruning, or feature selection occurs in this step.

In [ ]:
from posthoc_analysis.decoder_training import (
    preprocess_session5_training_subject,
    validate_session5_analysis_channel_layout,
)


In [ ]:
session5_channel_layout = validate_session5_analysis_channel_layout(
    session5_run_manifest
)
session5_channel_layout['analysis_eeg_labels']


In [ ]:
# Start with one participant for inspection; repeat for another participant only after review.
session5_preprocessed_e21 = preprocess_session5_training_subject(
    'e21',
    session5_model_inputs,
)


In [ ]:
display(session5_preprocessed_e21['run_summary'])
print('Epoch array:', session5_preprocessed_e21['epochs_time_channels_trials'].shape)
print('Time range (s):', session5_preprocessed_e21['time'][0], 'to', session5_preprocessed_e21['time'][-1])
display(session5_preprocessed_e21['trial_table'].head())


### Step 3 — Conventional P/PO difference inputs

Create fixed-orientation conventional `right − left` P/PO difference signals, then split them into right-distractor/no-distractor and left-distractor/no-distractor trial sets. This is still deterministic preprocessing: no xDAWN, balancing, pruning, feature ranking, or classifier fitting occurs here.

The historical source code selected its left and right P/PO channel sets independently in acquired-label order, producing non-homologous P-channel subtractions. This new post-hoc model deliberately uses explicit conventional pairs: P2−P1, P4−P3, P6−P5, P8−P7, PO4−PO3, PO6−PO5, and PO8−PO7.

In [ ]:
from posthoc_analysis.decoder_training import construct_session5_conventional_difference_inputs


In [ ]:
session5_difference_inputs_e21 = construct_session5_conventional_difference_inputs(
    session5_preprocessed_e21
)


In [ ]:
print('Conventional difference channels:', session5_difference_inputs_e21['difference_channel_labels'])
print('Right/no input:', session5_difference_inputs_e21['right']['epochs_time_channels_trials'].shape)
print('Left/no input:', session5_difference_inputs_e21['left']['epochs_time_channels_trials'].shape)
display(session5_difference_inputs_e21['right']['trial_table'].head())
display(session5_difference_inputs_e21['left']['trial_table'].head())


### Step 4 — Fold-local xDAWN implementation check

The function below fits xDAWN only on a provisional balanced training subset from e21's right-decoder data: Runs 2–4 provide the training data and Run 1 is held out. This is an implementation check, not cross-validation or model selection. The full balancing, leave-one-run-out loop, pruning, r² selection, and LDA fit will be added later.

In [ ]:
from posthoc_analysis.decoder_training import (
    apply_xdawn_filters,
    fit_xdawn_fold,
    select_poststimulus_feature_window,
)
import numpy as np


In [ ]:
# Provisional fold: Run 1 held out; randomly balance Runs 2–4 within run.
right_check = session5_difference_inputs_e21['right']
right_check_trials = right_check['trial_table']
right_check_labels = right_check['labels']
rng = np.random.default_rng(20260812)
xdawn_check_train_indices = []
for run_id in (2, 3, 4):
    run_indices = np.flatnonzero(right_check_trials['run_id'].to_numpy() == run_id)
    positive_indices = run_indices[right_check_labels[run_indices] == 1]
    negative_indices = run_indices[right_check_labels[run_indices] == 0]
    sampled_negative_indices = rng.choice(
        negative_indices, size=len(positive_indices), replace=False
    )
    xdawn_check_train_indices.extend(positive_indices.tolist())
    xdawn_check_train_indices.extend(sampled_negative_indices.tolist())
xdawn_check_train_indices = np.array(sorted(xdawn_check_train_indices), dtype=int)
xdawn_window_indices = select_poststimulus_feature_window(
    session5_difference_inputs_e21['time'], start_s=0.2
)
xdawn_check_fit = fit_xdawn_fold(
    right_check['epochs_time_channels_trials'][:, :, xdawn_check_train_indices],
    right_check_labels[xdawn_check_train_indices],
    xdawn_window_indices,
    n_components=2,
)
xdawn_check_projected = apply_xdawn_filters(
    right_check['epochs_time_channels_trials'],
    xdawn_check_fit['positive_class_filters_components_by_channels'],
)


In [ ]:
print('xDAWN window:', session5_difference_inputs_e21['time'][xdawn_window_indices[0]],
      'to', session5_difference_inputs_e21['time'][xdawn_window_indices[-1]], 's')
print('Fold training trials:', len(xdawn_check_train_indices))
print('Class-1 filters (components × channels):',
      xdawn_check_fit['positive_class_filters_components_by_channels'].shape)
print('Projected right/no data (time × components × trials):',
      xdawn_check_projected.shape)


### Step 5 — Reproducible within-run balancing

Balance the binary classes separately within the fold's training runs. The check below holds out e21 Run 1 and balances Runs 2–4 only, leaving the held-out run completely unselected. The returned full-length mask and audit table will be reused inside each fold of iterative pruning.

In [ ]:
from posthoc_analysis.decoder_training import balance_binary_trials_within_run


In [ ]:
right_balance_check = balance_binary_trials_within_run(
    session5_difference_inputs_e21['right']['trial_table'],
    session5_difference_inputs_e21['right']['labels'],
    training_run_ids=(2, 3, 4),  # Run 1 is held out.
    random_seed=20260812,
)
display(right_balance_check['audit'])


In [ ]:
right_check_trials = session5_difference_inputs_e21['right']['trial_table']
right_check_labels = session5_difference_inputs_e21['right']['labels']
selected = right_balance_check['selected_mask']
print('Selected trials in held-out Run 1:', int(selected[right_check_trials['run_id'].to_numpy() == 1].sum()))
print('Selected class counts:', np.bincount(right_check_labels[selected], minlength=2))


### Step 6 — Fold-local feature construction and r² ranking

Using the balanced e21 Runs 2–4 training subset and untouched Run 1 held-out data, fit xDAWN, project both datasets with the fold-specific class-1 filters, stride-subsample the 200 ms–final-sample window at ratio 8, fit z-scoring only on training features, and rank candidate features by training r². This is still an implementation check: no LDA, posterior calibration, AUPRC, or pruning is calculated here.

In [ ]:
from posthoc_analysis.decoder_training import fit_fold_feature_pipeline


In [ ]:
right_check_epochs = session5_difference_inputs_e21['right']['epochs_time_channels_trials']
right_check_trials = session5_difference_inputs_e21['right']['trial_table']
heldout_run1 = right_check_trials['run_id'].to_numpy() == 1
balanced_training = right_balance_check['selected_mask']
right_feature_check = fit_fold_feature_pipeline(
    right_check_epochs[:, :, balanced_training],
    right_check_labels[balanced_training],
    right_check_epochs[:, :, heldout_run1],
    session5_difference_inputs_e21['time'],
    feature_start_s=0.2,
    resample_ratio=8,
    n_xdawn_components=2,
    n_selected_features=30,
)


In [ ]:
print('Candidate features:', len(right_feature_check['r2']))
print('Selected training matrix:', right_feature_check['training_selected_features'].shape)
print('Selected held-out matrix:', right_feature_check['heldout_selected_features'].shape)
display(right_feature_check['selected_coordinates'])


### Step 7 — Fold-local regularized LDA and posterior calibration

Fit the source-style regularized linear LDA only on the 30 selected, balanced training features. The covariance uses `Gamma = 0.05` and uniform priors. Convert its distances to posterior probabilities with the original 2.5th/97.5th-quantile sigmoid rule, then inspect only the untouched held-out Run 1 predictions. AUPRC and pruning are still not calculated in this step.

In [ ]:
from posthoc_analysis.decoder_training import fit_fold_lda_and_calibrate


In [ ]:
right_lda_check = fit_fold_lda_and_calibrate(
    right_feature_check,
    right_check_labels[balanced_training],
    gamma=0.05,
)


In [ ]:
heldout_right_labels = right_check_labels[heldout_run1]
heldout_right_posteriors = right_lda_check['heldout_prediction']['posterior']
print('Held-out class counts:', np.bincount(heldout_right_labels, minlength=2))
print('Held-out posterior range:', heldout_right_posteriors.min(), 'to', heldout_right_posteriors.max())
print('Calibration slope:', right_lda_check['calibration']['slope'])


### Step 8 — Full leave-one-run-out prediction pass and uniform-prior PR-AUC

Fit four independent folds for e21's right/no model. In each fold, balance only the three training runs, fit all learned transforms exclusively there, and score the complete held-out run. Combine the 180 held-out predictions and calculate the MATLAB-style uniform-prior PR-AUC. This is the unpruned baseline iteration; iterative pruning has not yet started.

In [ ]:
from posthoc_analysis.decoder_training import run_leave_one_run_out_feature_cv


In [ ]:
right_cv_iteration0_e21 = run_leave_one_run_out_feature_cv(
    session5_difference_inputs_e21['right'],
    session5_difference_inputs_e21['time'],
    random_seed=20260812,
    feature_start_s=0.2,
    resample_ratio=8,
    n_xdawn_components=2,
    n_selected_features=30,
    gamma=0.05,
)


In [ ]:
display(right_cv_iteration0_e21['fold_audit'])
print('Uniform-prior PR-AUC:', right_cv_iteration0_e21['uniform_prior_pr']['auc'])
display(right_cv_iteration0_e21['selected_features_by_fold'].head())


### Step 9 — Cumulative iterative-pruning check

Run the first three source-style pruning iterations for e21's right/no model as an inspectable check. Each iteration balances all currently active runs once, reuses that shared balanced pool across the four folds, scores every original trial in its held-out run, then removes high-confidence errors and near-0.5 posteriors from the active mask. The reusable function defaults to the source’s full 20 iterations; do not interpret this three-iteration check as the final selected clean dataset.

In [ ]:
from posthoc_analysis.decoder_training import run_iterative_pruning_feature_cv


In [ ]:
right_pruning_check_e21 = run_iterative_pruning_feature_cv(
    session5_difference_inputs_e21['right'],
    session5_difference_inputs_e21['time'],
    n_iterations=3,
    random_seed=20260812,
    feature_start_s=0.2,
    resample_ratio=8,
    n_xdawn_components=2,
    n_selected_features=30,
    gamma=0.05,
)


In [ ]:
display(right_pruning_check_e21['history'])
display(right_pruning_check_e21['pruning_audit'])
print('Best check iteration:', right_pruning_check_e21['best_iteration'])
print('Clean trials at its start:', int(right_pruning_check_e21['best_mask'].sum()))


### Step 10 — Full 20-iteration clean-trial selection

After inspecting the three-iteration check, run the source-matched full 20 iterations for e21's right/no model. The best iteration is selected by pooled uniform-prior PR-AUC; its mask is the clean Session 5 training subset that will be used only in the subsequent final refit.

In [ ]:
right_pruning_full_e21 = run_iterative_pruning_feature_cv(
    session5_difference_inputs_e21['right'],
    session5_difference_inputs_e21['time'],
    n_iterations=20,
    random_seed=20260812,
    feature_start_s=0.2,
    resample_ratio=8,
    n_xdawn_components=2,
    n_selected_features=30,
    gamma=0.05,
)


In [ ]:
display(right_pruning_full_e21['history'])
print('Selected iteration:', right_pruning_full_e21['best_iteration'])
print('Selected clean trials:', int(right_pruning_full_e21['best_mask'].sum()))
display(right_pruning_full_e21['best_trial_table'].head())


### Step 11 — Final clean-trial Session 5 feature reference

Refit only the transform needed for longitudinal feature tracking on the best clean right/no dataset: xDAWN, temporal resampling, training-data z-scoring, and r² ranking. As in the original final refit, use every retained clean trial without additional class balancing. Do not fit a final classifier here; the output is the frozen xDAWN weights, normalization parameters, and top-30 feature coordinates for later application to Sessions 1–5.

In [ ]:
from posthoc_analysis.decoder_training import fit_final_clean_feature_reference


In [ ]:
right_clean_mask = right_pruning_full_e21['best_mask']
right_clean_epochs = session5_difference_inputs_e21['right']['epochs_time_channels_trials'][:, :, right_clean_mask]
right_clean_labels = session5_difference_inputs_e21['right']['labels'][right_clean_mask]
right_final_feature_reference_e21 = fit_final_clean_feature_reference(
    right_clean_epochs,
    right_clean_labels,
    session5_difference_inputs_e21['time'],
    feature_start_s=0.2,
    resample_ratio=8,
    n_xdawn_components=2,
    n_selected_features=30,
)


In [ ]:
print('Frozen xDAWN filters:', right_final_feature_reference_e21['xdawn_filters_components_by_channels'].shape)
print('Frozen feature indices:', right_final_feature_reference_e21['selected_indices_zero_based'])
display(right_final_feature_reference_e21['selected_coordinates'])


### Step 12 — Full unpruned Session 5 training-data r² reference

Apply the frozen right/no feature reference to all eligible, unpruned Session 5 right/no training trials. Recalculate r² for the already selected 30 coordinates without refitting xDAWN, z-scoring, or feature selection. This reduces pruning-related inflation but remains selection-linked because the feature coordinates originated from this same Session 5 training task; Session 5 decoding runs will provide the independent post-training comparison.

In [ ]:
from posthoc_analysis.decoder_training import apply_frozen_feature_reference_and_compute_r2


In [ ]:
right_full_session5_r2_e21 = apply_frozen_feature_reference_and_compute_r2(
    session5_difference_inputs_e21['right']['epochs_time_channels_trials'],
    session5_difference_inputs_e21['right']['labels'],
    right_final_feature_reference_e21,
)


In [ ]:
right_full_session5_r2_table_e21 = right_full_session5_r2_e21['feature_r2_table'].copy()
display(right_full_session5_r2_table_e21)
print('Mean fixed-feature r² on full Session 5 training data:',
      right_full_session5_r2_table_e21['r2_evaluation'].mean())


### Step 13 — Validate longitudinal evaluation inputs

Before applying a frozen Session 5 reference, validate the intended evaluation datasets separately: eight Session 1 training runs as the pre-intervention baseline, plus all non-practice decoding runs from Sessions 1–5. Decoding and training analysis files have different valid BCI-output values, so they are validated with task-specific rules. This step also validates 60 stimulus events, trigger/Status condition order, clock conversion, complete epoch bounds, and right/no plus left/no labels for every run.

In [ ]:
from posthoc_analysis.decoder_training import build_longitudinal_evaluation_manifest


In [ ]:
# Start with e21 for review. The same function accepts no subject_ids argument to validate all participants.
longitudinal_inputs_e21 = build_longitudinal_evaluation_manifest(
    subject_ids=['e21'],
    project_root=PROJECT_ROOT,
)
longitudinal_run_manifest_e21 = longitudinal_inputs_e21['manifest']


In [ ]:
display(longitudinal_inputs_e21['summary'])
display(longitudinal_inputs_e21['issues'])
display(
    longitudinal_run_manifest_e21[
        [
            'evaluation_task', 'session_id', 'run_id', 'n_trials',
            'n_no_distractor', 'n_distractor_right', 'n_distractor_left',
            'status_to_trigger_time_slope',
            'max_trigger_time_residual_samples', 'status_event_alignment',
        ]
    ]
)


### Step 14 — Preprocess longitudinal evaluation EEG

Apply the same post-hoc preprocessing to the validated Session 1 training baseline and all Session 1–5 decoding runs: label-derived 61 scalp EEG channels, zero-phase 0.1–20 Hz FIR filtering, Status-anchored −0.5 to 1.0 s epochs, and −0.2 to 0 s per-trial baseline correction. No xDAWN, normalization, feature ranking, or classifier is fit in this step. The output remains ordered as pre-intervention training first, followed by decoding Sessions 1–5.


In [ ]:
from posthoc_analysis.decoder_training import preprocess_longitudinal_evaluation_subject


In [ ]:
# Review one participant first. Replace 'e21' only after inspecting the checks below.
longitudinal_preprocessed_e21 = preprocess_longitudinal_evaluation_subject(
    'e21',
    longitudinal_inputs_e21,
)


In [ ]:
display(longitudinal_preprocessed_e21['run_summary'].groupby(
    ['evaluation_task', 'session_id'], as_index=False
).agg(
    n_runs=('run_id', 'size'),
    max_abs_residual_baseline_mean=('max_abs_residual_baseline_mean', 'max'),
))
print('Epoch array (time × channels × trials):',
      longitudinal_preprocessed_e21['epochs_time_channels_trials'].shape)
print('EEG channels:', longitudinal_preprocessed_e21['eeg_labels'])


### Step 15 — Full-cohort longitudinal preprocessing QC

Process each participant sequentially using the validated longitudinal manifest. This is a QC pass only: it validates channel layout, Status-event/trial alignment, epoch dimensions, and residual baseline means, then releases the epoch array before processing the next participant. No model, xDAWN transform, normalization, or r² value is fitted here.


In [ ]:
from posthoc_analysis.decoder_training import (
    build_longitudinal_evaluation_manifest,
    run_longitudinal_preprocessing_qc,
)


In [ ]:
# Rebuild after any documented run-availability decision, then QC all participants.
longitudinal_inputs = build_longitudinal_evaluation_manifest(project_root=PROJECT_ROOT)
longitudinal_preprocessing_qc = run_longitudinal_preprocessing_qc(
    longitudinal_inputs,
)
display(longitudinal_preprocessing_qc['subject_qc'])
display(longitudinal_preprocessing_qc['failures'])


### Step 16 — Checkpointed full-cohort QC

Run each participant in a fresh Python process and checkpoint the subject-level, run-level, and failure tables after every participant. This is the recommended cohort-scale QC path because it prevents MNE memory accumulation and can resume after interruption.


In [ ]:
from posthoc_analysis.decoder_training import run_checkpointed_longitudinal_preprocessing_qc


In [ ]:
longitudinal_qc = run_checkpointed_longitudinal_preprocessing_qc(
    project_root=PROJECT_ROOT,
    output_dir=repo_root / 'analyses' / 'qc',
    resume=True,
)
display(longitudinal_qc['subject_qc'])
display(longitudinal_qc['failures'])
print(longitudinal_qc['paths'])


### Step 17 — Build one participant's Session-5 top-30 feature references

Build fresh right/no and left/no references from the four Session-5 training runs only. Each side uses conventional P/PO differences, fold-local xDAWN, run-wise balancing, leave-one-run-out iterative pruning, and a final clean-trial top-30 r² refit. This first cell builds e21 for inspection; no decoding-session data enter selection.


In [ ]:
from posthoc_analysis.decoder_training import (
    build_session5_training_model_manifest,
    build_session5_top30_feature_references_for_subject,
)


In [ ]:
session5_inputs_e21 = build_session5_training_model_manifest(
    subject_ids=['e21'],
    project_root=PROJECT_ROOT,
)
session5_references_e21 = build_session5_top30_feature_references_for_subject(
    'e21',
    session5_inputs_e21,
)


In [ ]:
display(session5_references_e21['summary'])
display(session5_references_e21['top30_features'])
display(session5_references_e21['clean_trials'])


### Step 18 — Save the validated frozen reference tables

After inspecting the e21 result, save the compact reference artifacts. The Parquet files retain clean-trial identities, final clean-training r² values, and selection summaries; per-decoder `.npz` artifacts retain the final Session-5 xDAWN filters, resampling indices, z-score parameters, and selected feature indices. These transforms are applied unchanged—not refit—to Session-1 and decoding data. Re-running this cell for a participant safely replaces that participant’s rows.


In [ ]:
from posthoc_analysis.decoder_training import (
    save_session5_top30_feature_references,
    load_session5_frozen_feature_reference,
)


In [ ]:
saved_session5_references_e21 = save_session5_top30_feature_references(
    session5_references_e21,
    output_dir=repo_root / 'analyses' / 'session5_feature_references',
)
print(saved_session5_references_e21['paths'])


In [ ]:
# Verify that the persisted transform matches the in-memory final Session-5 reference.
e21_right_frozen_reference = load_session5_frozen_feature_reference(
    'e21', 'right', output_dir=repo_root / 'analyses' / 'session5_feature_references'
)
assert (e21_right_frozen_reference['selected_indices_zero_based'] ==
        session5_references_e21['side_results']['right']['reference']['selected_indices_zero_based']).all()
assert (e21_right_frozen_reference['xdawn_filters_components_by_channels'] ==
        session5_references_e21['side_results']['right']['reference']['xdawn_filters_components_by_channels']).all()
print('Persisted right-decoder frozen transform matches the final Session-5 fit.')


### Step 19 — Checkpointed cohort build of frozen Session-5 references

Build the right/no and left/no frozen references for every participant. The validated Session-5 manifest is prepared once; each participant is then checkpointed immediately. With `resume=True`, existing complete references are validated and skipped, while incomplete references are rebuilt. The retained build log records selection outcomes and any subject-specific failure.


In [ ]:
from posthoc_analysis.config import PROJECT_ROOT
from posthoc_analysis.decoder_training import run_checkpointed_session5_reference_build


In [ ]:
session5_reference_cohort_build = run_checkpointed_session5_reference_build(
    project_root=PROJECT_ROOT,
    output_dir=repo_root / 'analyses' / 'session5_feature_references',
    n_pruning_iterations=20,
    random_seed=20260812,
    resume=True,
)
display(session5_reference_cohort_build['build_log'])
print(session5_reference_cohort_build['build_log_path'])


### Step 20 — Apply frozen Session-5 features to one participant's longitudinal EEG

For each right/no and left/no reference, evaluate Session-1 training and all eligible Session-1–5 decoding runs. Run-level r² uses one run’s trials. Session-level r² pools all eligible trials in a task × session before calculating r²; it is not an average of run-level r² values. The saved Session-5 xDAWN filters, normalization, and feature coordinates are applied unchanged.


In [ ]:
from posthoc_analysis.config import PROJECT_ROOT
from posthoc_analysis.decoder_training import (
    build_longitudinal_evaluation_manifest,
    evaluate_frozen_session5_features_for_subject,
    save_longitudinal_frozen_r2_for_subject,
)


In [ ]:
longitudinal_inputs_e21 = build_longitudinal_evaluation_manifest(
    subject_ids=['e21'],
    project_root=PROJECT_ROOT,
)
e21_longitudinal_frozen_r2 = evaluate_frozen_session5_features_for_subject(
    'e21',
    longitudinal_inputs_e21,
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)


In [ ]:
display(e21_longitudinal_frozen_r2['r2_table'])
display(e21_longitudinal_frozen_r2['r2_table'].groupby(
    ['aggregation_level', 'evaluation_task', 'session_id', 'decoder_side'], dropna=False
).agg(n_features=('rank', 'size'), n_trials=('n_binary_trials', 'first'),
      n_runs=('n_contributing_runs', 'first'), mean_r2=('r2_evaluation', 'mean')))


In [ ]:
saved_e21_longitudinal_frozen_r2 = save_longitudinal_frozen_r2_for_subject(
    e21_longitudinal_frozen_r2,
    output_dir=repo_root / 'analyses' / 'session5_feature_references',
)
print(saved_e21_longitudinal_frozen_r2['path'])


### Step 21 — Checkpointed cohort evaluation of frozen top-30 features

Apply every participant’s frozen Session-5 reference to validated Session-1 training and Session-1–5 decoding EEG. The cohort runner checkpoints each participant’s run-wise and trial-pooled session-wise r² rows immediately. With `resume=True`, it validates and skips an already complete participant rather than recomputing it.


In [ ]:
from posthoc_analysis.decoder_training import run_checkpointed_longitudinal_frozen_r2_evaluation


In [ ]:
longitudinal_frozen_r2_cohort = run_checkpointed_longitudinal_frozen_r2_evaluation(
    project_root=PROJECT_ROOT,
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
    resume=True,
)
display(longitudinal_frozen_r2_cohort['build_log'])
print(longitudinal_frozen_r2_cohort['r2_path'])


### Step 22 — Visualize stability of the Session-5 top-30 feature sets

For each group and decoder, calculate the percentage of participants who selected each candidate component × time coordinate among their top 30 features. The x-axis shows the complete stride-8 candidate time grid, with a deliberate gap between xDAWN Components 1 and 2. Candidate coordinates selected by no participant remain visible at 0%.


In [ ]:
from posthoc_analysis.decoder_training import (
    summarize_session5_top30_selection_frequency,
    plot_session5_top30_selection_frequency,
)


In [ ]:
session5_selection_frequency = summarize_session5_top30_selection_frequency(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)
session5_selection_frequency_figure = plot_session5_top30_selection_frequency(
    session5_selection_frequency,
    output_dir=repo_root / 'figures',
)


In [ ]:
display(session5_selection_frequency)
print(session5_selection_frequency_figure['pdf_path'])
print(session5_selection_frequency_figure['png_path'])


### Step 23 — Append full-dataset Session-5 training r² as the post-training reference

Apply each participant’s frozen Session-5 reference to all valid trials from the four non-practice Session-5 training runs. This produces run-wise and pooled-session r² for the fixed top-30 features without refitting any transform. These values are selection-linked descriptive post-training measures because the feature set was selected from this dataset; they are not independent model-performance estimates. The append validates every participant before atomically updating the retained longitudinal r² table.


In [ ]:
from posthoc_analysis.decoder_training import append_session5_training_post_frozen_r2


In [ ]:
session5_training_post_append = append_session5_training_post_frozen_r2(
    project_root=PROJECT_ROOT,
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
    persist=True,
)
display(session5_training_post_append['append_log'])
print(session5_training_post_append['r2_path'])


### Step 24 — Track mean frozen-feature r² across the training and intervention timeline

For each participant, decoder, and assessment, average the trial-pooled r² values across the 30 frozen Session-5 features. The complete timeline comprises full-dataset Session-1 training (pre-training), decoding on intervention days 1–5, and full-dataset Session-5 training (post-training). Then calculate group means and between-participant SEM. Right and left decoders are plotted separately because they use distinct feature sets and distinguish different distractor classes. The post-training training value is selection-linked and descriptive because the features were selected from Session-5 training.


In [ ]:
from posthoc_analysis.decoder_training import (
    summarize_longitudinal_top30_r2_by_session,
    plot_longitudinal_top30_r2_by_session,
)


In [ ]:
longitudinal_top30_r2_session_summary = summarize_longitudinal_top30_r2_by_session(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)
longitudinal_top30_r2_session_figure = plot_longitudinal_top30_r2_by_session(
    longitudinal_top30_r2_session_summary,
    output_dir=repo_root / 'figures',
)


In [ ]:
display(longitudinal_top30_r2_session_summary['group_session'])
print(longitudinal_top30_r2_session_figure['pdf_path'])
print(longitudinal_top30_r2_session_figure['png_path'])


### Step 25 — Combine right and left decoder r² trajectories

For each participant and assessment, first retain the mean r² across the 30 frozen features separately for the right and left decoder. Average these two decoder means with equal weight, then calculate group means and between-participant SEM. This single-panel summary treats both lateralized decoder feature sets equally.


In [ ]:
from posthoc_analysis.decoder_training import (
    summarize_longitudinal_top30_r2_combined_decoders,
    plot_longitudinal_top30_r2_combined_decoders,
)


In [ ]:
longitudinal_top30_r2_combined_summary = summarize_longitudinal_top30_r2_combined_decoders(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)
longitudinal_top30_r2_combined_figure = plot_longitudinal_top30_r2_combined_decoders(
    longitudinal_top30_r2_combined_summary,
    output_dir=repo_root / 'figures',
)


In [ ]:
display(longitudinal_top30_r2_combined_summary['group_assessment'])
print(longitudinal_top30_r2_combined_figure['pdf_path'])
print(longitudinal_top30_r2_combined_figure['png_path'])


### Step 26 — Repeat longitudinal r² plots for the top 10 frozen features

Restrict the frozen feature set to ranks 1–10, which are the ten strongest Session-5 selected features for each participant and decoder. Repeat the decoder-specific and equal-weighted combined-decoder trajectories using the same seven-assessment timeline and group SEM.


In [ ]:
top10_decoder_summary = summarize_longitudinal_top30_r2_by_session(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
    n_features=10,
)
top10_decoder_figure = plot_longitudinal_top30_r2_by_session(
    top10_decoder_summary,
    output_dir=repo_root / 'figures',
)


In [ ]:
top10_combined_summary = summarize_longitudinal_top30_r2_combined_decoders(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
    n_features=10,
)
top10_combined_figure = plot_longitudinal_top30_r2_combined_decoders(
    top10_combined_summary,
    output_dir=repo_root / 'figures',
)


In [ ]:
display(top10_decoder_summary['group_session'])
display(top10_combined_summary['group_assessment'])
print(top10_decoder_figure['pdf_path'])
print(top10_decoder_figure['png_path'])
print(top10_combined_figure['pdf_path'])
print(top10_combined_figure['png_path'])


### Step 27 — Test the combined-decoder top-30 r² trajectory difference between groups

Fit a linear mixed-effects model to the participant-level, equally weighted combined-decoder mean r² on decoding-based intervention days 1–5 only. The outcome is log r². Fixed effects are centered intervention day, group, and their interaction; the interaction directly tests whether BCI and mental rehearsal have different r² slopes. A participant random day slope is used when supported; otherwise, the prespecified random-intercept fallback is used. Pre-training and post-training task values are excluded from inference because the post-training value is selection-linked.


In [ ]:
from posthoc_analysis.decoder_training import run_top30_combined_decoder_intervention_slope_mixed_model


In [ ]:
top30_combined_decoder_slope_model = run_top30_combined_decoder_intervention_slope_mixed_model(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)
display(top30_combined_decoder_slope_model['effects'])
display(top30_combined_decoder_slope_model['fixed_effects'])
print(top30_combined_decoder_slope_model['model_structure'])


### Step 28 — Test decoder-specific top-30 r² trajectory differences

Fit a day × group × decoder-side mixed-effects model to log participant-level top-30 r² on decoding-based intervention days 1–5. The three-way interaction tests whether the BCI-versus-mental-rehearsal trajectory difference differs between right and left decoders. Planned BCI-minus-control slope contrasts are calculated separately for each decoder and Holm-corrected together.


In [ ]:
from posthoc_analysis.decoder_training import run_top30_decoder_specific_intervention_slope_mixed_model


In [ ]:
top30_decoder_specific_slope_model = run_top30_decoder_specific_intervention_slope_mixed_model(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)
display(top30_decoder_specific_slope_model['decoder_group_slope_contrasts'])
display(top30_decoder_specific_slope_model['three_way_interaction'])
display(top30_decoder_specific_slope_model['fixed_effects'])
print(top30_decoder_specific_slope_model['model_structure'])


### Step 29 — Mixed ANOVAs for combined, right, and left top-30 r²

Run separate 2-group × 5-day mixed ANOVAs on log participant mean r² for the combined-decoder, right-decoder, and left-decoder outcomes. Intervention day is categorical, allowing a non-linear pattern across training. Type-III Wald chi-square tests are calculated from participant-random-intercept mixed models. The combined-decoder model is primary; the two decoder-specific group × day interactions are Holm-corrected together.


In [ ]:
from posthoc_analysis.decoder_training import run_top30_intervention_day_mixed_anovas


In [ ]:
top30_intervention_day_anovas = run_top30_intervention_day_mixed_anovas(
    reference_dir=repo_root / 'analyses' / 'session5_feature_references',
)
display(top30_intervention_day_anovas['anova_table'])
display(top30_intervention_day_anovas['model_info'])


### Step 30 — Validate Session-5 decoding inputs for exploratory feature references

Build a separate, strict manifest for brand-new Session-5 decoding-based references. Every documented non-practice decoding run must validate before feature selection begins. This source dataset is distinct from the existing Session-5 training-task reference workflow.


In [ ]:
from posthoc_analysis.decoder_training import build_session5_decoding_model_manifest


In [ ]:
session5_decoding_model_inputs = build_session5_decoding_model_manifest(
    project_root=PROJECT_ROOT,
)
display(session5_decoding_model_inputs['group_summary'])
display(session5_decoding_model_inputs['manifest'].head())


### Step 31 — Preprocess and construct Session-5 decoding model inputs for one participant

Run the decoding-only source data through the fixed post-hoc EEG preprocessing pipeline, then construct conventional right-minus-left P/PO difference epochs and binary right/no and left/no datasets. Inspect this one-participant checkpoint before any fold-specific xDAWN fitting.


In [ ]:
from posthoc_analysis.decoder_training import (
    preprocess_session5_decoding_subject,
    construct_session5_conventional_difference_inputs,
)


In [ ]:
session5_decoding_smoke_subject = 'e21'
session5_decoding_smoke_preprocessed = preprocess_session5_decoding_subject(
    session5_decoding_smoke_subject, session5_decoding_model_inputs,
)
session5_decoding_smoke_difference_inputs = construct_session5_conventional_difference_inputs(
    session5_decoding_smoke_preprocessed,
)
display(session5_decoding_smoke_preprocessed['run_summary'])
print(session5_decoding_smoke_difference_inputs['difference_channel_labels'])


## Training-task post-minus-pre RT change by group and trial type

Summarize participant-level Session 5 minus Session 1 training-task reaction-time change separately for distractor and no-distractor trials. The analysis reuses the primary training RT inclusion rules and pooled participant × session ±3 SD outlier filtering. Negative values indicate faster post-intervention responding.


In [ ]:
from posthoc_analysis.behavioral import load_and_plot_training_rt_change_barplot


In [ ]:
training_rt_change_results = load_and_plot_training_rt_change_barplot()


In [ ]:
display(training_rt_change_results['change_scores'])
display(training_rt_change_results['change_summary'])
display(training_rt_change_results['subject_counts'])
display(training_rt_change_results['agreement_with_primary_rt'])


In [ ]:
print('Welch BCI-versus-Mental-Rehearsal RT-change tests (exact p-values and Cohen\'s d):')
display(training_rt_change_results['welch_tests'])
print('Saved PDF:', training_rt_change_results['figure_path'])
print('Saved PNG:', training_rt_change_results['png_path'])
training_rt_change_results['figure']


## Stroop post-minus-pre RT change by congruency

Create a poster-style grouped summary of participant-level Session 5 minus Session 1 Stroop reaction-time change for congruent and incongruent trials. This step reuses the existing Stroop Analysis 3 correct-trial inclusion and participant-session pooled ±3 SD outlier filtering across both runs and congruency conditions. Negative values indicate faster post-intervention responding.


Saved outputs in this earlier section used the historical per-run RT filter. Rerun to update statistics/figures; the appended **Stroop RT: pooled-session ±3 SD exclusions** section contains the current validated exclusion counts and RT summaries.


In [ ]:
from posthoc_analysis.behavioral import load_and_plot_stroop_rt_change_barplot


In [ ]:
stroop_rt_change_results = load_and_plot_stroop_rt_change_barplot()


In [ ]:
display(stroop_rt_change_results['change_scores'])
display(stroop_rt_change_results['change_summary'])
display(stroop_rt_change_results['subject_counts'])
display(stroop_rt_change_results['agreement_with_primary_analysis3'])


In [ ]:
print('Welch BCI-versus-Mental-Rehearsal Stroop RT-change tests (exact p-values and Cohen\'s d):')
display(stroop_rt_change_results['welch_tests'])
print('Saved PDF:', stroop_rt_change_results['figure_path'])
print('Saved PNG:', stroop_rt_change_results['png_path'])
stroop_rt_change_results['figure']


# Training RT clock correction audit
The consolidated training CSV above was reconstructed from GDF Status; the original RT is preserved in `gdf_rt_ms`, and absent responses with text RT <50 ms are explicitly pooled into `rt_ms` for fast-response reporting. The existing analysis sections apply correct, nonmissing RT ≥150 ms followed by participant/session pooled ±3 sample SD, then regenerate training RT, distractor-cost, RT-change, and Pd mean-RT reference-line figures. Stroop is unchanged.

This audit reuses `result` from **Consolidated Trial-wise Data**; it does not rebuild or clean the trials a second time.


In [ ]:
corrected_training_trials = result["dataframe"]
display(corrected_training_trials[[
    "subject_id", "session_id", "run_id", "trial", "rt_ms", "gdf_rt_ms", "text_rt_ms",
    "rt_samples", "rt_source", "response_count", "alignment_error",
]].head())
print("Rebuilt CSV:", result["output_path"])
print("Runs / trials:", result["total_runs"], result["total_trials"])
print("Missing GDF RTs:", corrected_training_trials["gdf_rt_ms"].isna().sum())
print("Multiple-response alignment errors:", corrected_training_trials["alignment_error"].sum())

print("Text RT fallbacks:", corrected_training_trials["rt_source"].eq("trigger_text_fallback").sum())
print("Unresolved analysis RTs:", corrected_training_trials["rt_ms"].isna().sum())


# Missing training GDF responses: raw Status audit
Investigate the 47 missing responses by subject/session/run/trial. Reuse the corrected full-cohort trial table, revalidate affected GDF/text/analysis files, inspect every Status sample in each missing-response interval, and compare missingness against diagnostic text RT. Missingness is based on GDF response counts even when analysis `rt_ms` has an explicit text fallback. Original `gdf_rt_ms` remains missing.

The supplied experiment branch initializes a 50 ms trigger timer, rejects overlapping hardware requests, and logs them to text regardless of success. Its input loop also permits queued ITI/fixation inputs to be accepted after onset. Source references, evidence limits, and the full findings are in [training_response_qc.md](../doc/training_response_qc.md). This raw audit does not change RT values; text pooling is handled by consolidation. Cleaning and figures remain unchanged.

In [ ]:
from posthoc_analysis.rt_qc import audit_missing_training_responses


In [ ]:
missing_response_qc = audit_missing_training_responses(
    corrected_training_trials,
    output_dir=project_root / "analyses" / "qc",
)


In [ ]:
display(missing_response_qc["by_text_rt"])
display(missing_response_qc["by_group_session"])
display(missing_response_qc["missing_trials"][[
    "subject_id", "session_id", "run_id", "trial", "feedback",
    "stimulus_trigger", "text_rt_ms", "raw_64_sample_count",
    "raw_response_bit_sample_count", "alternative_extraction_response_count",
]])


# Training RT text fallback and fast-response reporting
Pool the audited missing responses from aligned trigger-text milliseconds into `rt_ms`, preserving `gdf_rt_ms` and missing GDF sample indices. Only absent GDF responses with text RT <50 ms qualify; ambiguous responses and other missing-response patterns remain missing.

All 47 fallbacks fail the existing 150 ms minimum. Descriptive fast-RT counts include correct and incorrect responses (53 total). The sequential exclusion report counts only correct fast trials under <150 ms (27); incorrect trials are already counted under non-correct responses. This avoids double-counting and leaves retained trials and pooled ±3 sample SD cleaning unchanged.

In [ ]:
from posthoc_analysis.triggers import apply_training_text_rt_fallback

In [ ]:
# Idempotent provenance validation of the table already built above.
fallback_audit = apply_training_text_rt_fallback(corrected_training_trials)
print("RT sources:")
display(fallback_audit["rt_source"].value_counts().rename("n_trials").to_frame())
print("All RTs <150 ms, by source and feedback (1=correct, 2=incorrect, 3=timeout):")
display(fallback_audit.loc[fallback_audit.rt_ms < 150]
        .groupby(["rt_source", "feedback"]).size().rename("n_trials").to_frame())
print("Sequential exclusions (correct fast trials only; incorrect trials counted separately):")
display(analysis1_results["subject_session_exclusion_report"][[
    "n_removed_non_correct", "n_removed_rt_lt_150", "n_removed_missing_rt",
    "n_retained_before_sd", "n_outliers_removed",
]].sum().rename("n_trials").to_frame())


## Distractor-task RT: 2 × 2 × 2 mixed ANOVA

Reanalyse the existing cleaned mean RTs with Group (BCI/control) between participants and Session (1/5) × Distractor Condition (absent/present) within participants. Reuse `analysis1_results` from Analysis 1; no exclusions, trial averaging, figures, or planned post hoc contrasts change. Each within effect uses its own participant-within-group error stratum. All effects have numerator df=1, so sphericity correction is unnecessary. This replaces the earlier shared-residual OLS analysis; Analysis 1 now also calls this mixed-ANOVA function when rerun.


In [ ]:
import importlib
import pandas as pd
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import run_training_rt_mixed_anova


In [ ]:
rt_summary_before_mixed_anova = analysis1_results["subject_session_summary"].copy(deep=True)
training_rt_mixed_anova_results = run_training_rt_mixed_anova(
    analysis1_results["subject_session_summary"]
)
pd.testing.assert_frame_equal(
    analysis1_results["subject_session_summary"], rt_summary_before_mixed_anova
)
analysis1_results["stats_anova"] = training_rt_mixed_anova_results
print("Retained RT trials (unchanged):", len(analysis1_results["trial_data_clean"]))
print("Participant-level RT means unchanged; only the ANOVA was replaced.")


In [ ]:
display(training_rt_mixed_anova_results["anova_report"])
display(training_rt_mixed_anova_results["error_terms"])
display(analysis1_results["cell_summary"])
training_rt_mixed_anova_results["diagnostics"]


## Distractor-task accuracy: exclude RT <150 ms, 2 × 2 × 2 mixed ANOVA

Using the original corrected training trials from the RT-clock reconstruction section, first remove RTs below 150 ms (both correct and incorrect responses). Compute correct/(correct + incorrect) per participant × session × condition after removing timeouts. Retain exactly 150-ms and slow responses; do not apply the correct-only or ±3-SD RT filters. Any missing RT is not classified as fast and is explicitly counted/warned about; there are none in the current corrected table.

Run Group (BCI/control, between participants) × Session (1/5, within participants) × Distractor Condition (present/absent, within participants), using the same separate error strata as the revised RT ANOVA. This section supersedes the historical unfiltered accuracy results above. Existing RT results remain unchanged.


In [ ]:
import importlib
import pandas as pd
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import (
    summarize_training_accuracy_after_rt_filter,
    run_training_accuracy_mixed_anova,
)


In [ ]:
training_accuracy_mixed_results = summarize_training_accuracy_after_rt_filter(
    corrected_training_trials, min_rt_ms=150.0
)
print("Sequential exclusions and denominator totals:")
display(training_accuracy_mixed_results["exclusion_report"][[
    "n_total_trials", "n_removed_rt_lt_cutoff", "n_timeouts_after_rt_filter",
    "n_accuracy_trials", "n_missing_rt_retained",
]].sum().rename("n_trials").to_frame())


In [ ]:
training_accuracy_mixed_results["stats_anova"] = run_training_accuracy_mixed_anova(
    training_accuracy_mixed_results["subject_session_summary"]
)


In [ ]:
with pd.option_context("display.max_columns", None):
    display(training_accuracy_mixed_results["cell_summary"])
    display(training_accuracy_mixed_results["stats_anova"]["anova_report"])
    display(training_accuracy_mixed_results["stats_anova"]["error_terms"])
display(training_accuracy_mixed_results["stats_anova"]["diagnostics"])


## Stroop accuracy: 2 × 2 × 2 mixed ANOVA

Test Group (BCI/mental-rehearsal control) × Session (pre/post) × Congruency (congruent/incongruent), with Group between participants and both other factors within participants. Reuse the original trial table from Stroop Analysis 2. Accuracy remains correct/(correct + incorrect), with timeouts excluded and no RT cutoff or RT-outlier filtering. Participant summaries pool trial counts across non-practice runs. The shared mixed-ANOVA engine uses effect-specific participant errors and reports all seven effects with partial η²; it requires complete four-cell coverage and balanced groups. This replaces the earlier separate accuracy OLS analyses.


In [ ]:
import importlib
import pandas as pd
from IPython.display import display
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import (
    summarize_stroop_accuracy,
    run_stroop_accuracy_mixed_anova,
)


In [ ]:
stroop_accuracy_mixed_results = summarize_stroop_accuracy(
    stroop_analysis2_results["original_data"]
)
display(stroop_accuracy_mixed_results["exclusion_report"])


In [ ]:
stroop_accuracy_mixed_results["stats_anova"] = run_stroop_accuracy_mixed_anova(
    stroop_accuracy_mixed_results["subject_session_summary"]
)


In [ ]:
with pd.option_context("display.max_columns", None):
    display(stroop_accuracy_mixed_results["cell_summary"])
    display(stroop_accuracy_mixed_results["stats_anova"]["anova_report"])
    display(stroop_accuracy_mixed_results["stats_anova"]["error_terms"])
display(stroop_accuracy_mixed_results["stats_anova"]["diagnostics"])


## Stroop RT: pooled-session ±3 SD exclusions

Replace per-run thresholds with one mean ±3 sample-SD threshold per participant and session, pooling correct-response trials across both non-practice runs and congruency conditions. Apply a single pass, with no minimum RT cutoff. The three exclusion categories (timeouts, incorrect responses and SD outliers) are mutually exclusive. Overall removal percentages use all non-practice trials as the denominator. Reuse the original Stroop table, and keep the cleaned trials and participant summaries for subsequent analysis.


In [ ]:
import importlib
from IPython.display import display
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import summarize_stroop_reaction_time


In [ ]:
stroop_rt_pooled_results = summarize_stroop_reaction_time(
    stroop_analysis2_results["original_data"]
)
display(stroop_rt_pooled_results["exclusion_breakdown"])


In [ ]:
display(stroop_rt_pooled_results["outlier_report"])
display(stroop_rt_pooled_results["subject_exclusion_summary"])
display(stroop_rt_pooled_results["subject_session_summary"])


## Stroop RT: 2 × 2 × 2 mixed ANOVA

Test Group (BCI/mental-rehearsal control) × Session (pre/post) × Congruency (congruent/incongruent), with Group between participants and the other factors within participants. Reuse the participant means from `stroop_rt_pooled_results`: correct responses only, with one participant-session ±3 sample-SD filter pooling both runs and congruency conditions, and no minimum RT cutoff. Exclude descriptive overall rows from the ANOVA. No additional trial exclusions or averaging occur here. The balanced mixed ANOVA tests all seven effects against the appropriate participant error terms and reports partial η². Group-cell confidence intervals are descriptive 95% t intervals.


In [ ]:
import importlib
import pandas as pd
from IPython.display import display
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import run_stroop_rt_mixed_anova


In [ ]:
stroop_rt_mixed_summary = stroop_rt_pooled_results["subject_session_summary"].loc[
    lambda table: table["trial_type"].isin(["congruent", "incongruent"])
].copy()
stroop_rt_mixed_stats = run_stroop_rt_mixed_anova(stroop_rt_mixed_summary)


In [ ]:
with pd.option_context("display.max_columns", None):
    display(stroop_rt_mixed_stats["cell_summary"])
    display(stroop_rt_mixed_stats["anova_report"])
    display(stroop_rt_mixed_stats["error_terms"])
display(stroop_rt_mixed_stats["diagnostics"])
display(stroop_rt_pooled_results["exclusion_breakdown"])


## Stroop RT: BCI versus control change by congruency

Compare pre–post RT changes between BCI and mental-rehearsal groups separately for congruent and incongruent trials. Reuse the participant RT means from the updated mixed ANOVA, with no further trial exclusions. The contrast is (BCI post−pre) − (control post−pre), so negative values indicate a larger RT decrease in BCI. Two-sided Welch tests use participant change scores; P values are unadjusted; no multiple-comparison correction is applied, as requested. Confidence intervals are pointwise unadjusted 95% Welch intervals. These secondary tests do not establish that the between-group change differs by congruency; that is the three-way ANOVA interaction.


In [ ]:
import importlib
import pandas as pd
from IPython.display import display
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import compare_stroop_rt_changes_by_congruency


In [ ]:
stroop_rt_group_change_results = compare_stroop_rt_changes_by_congruency(
    stroop_rt_mixed_summary
)


In [ ]:
with pd.option_context("display.max_columns", None):
    display(stroop_rt_group_change_results["welch_tests"])
    display(stroop_rt_group_change_results["change_scores"])


## Distractor RT: BCI versus control change by condition

Compare participant post-minus-pre RT changes between BCI and mental rehearsal separately for distractor-present and distractor-absent trials. Reuse the same participant means as the current distractor RT mixed ANOVA: correct, nonmissing RT ≥150 ms, followed by a single ±3 sample-SD filter within participant/session pooling all runs and both conditions. Apply no additional exclusions. Two-sided Welch independent-samples tests report unadjusted p values, 95% Welch confidence intervals, and Cohen's d on change scores. Negative BCI-minus-control differences indicate greater RT reduction in BCI. These tests do not establish a difference between conditions; that question is addressed by the three-way interaction.


In [ ]:
import importlib
import pandas as pd
from IPython.display import display
import posthoc_analysis.behavioral as behavioral
importlib.reload(behavioral)
from posthoc_analysis.behavioral import compare_training_rt_changes_by_condition


In [ ]:
training_rt_group_change_results = compare_training_rt_changes_by_condition(
    analysis1_results["subject_session_summary"]
)


In [ ]:
with pd.option_context("display.max_columns", None):
    display(training_rt_group_change_results["welch_tests"])
    display(training_rt_group_change_results["change_scores"])
    display(analysis1_results["cell_summary"])


## Exploratory analysis: previous BCI feedback and next-trial RT

Test the association between feedback on trial n and correct-response RT on trial n+1, pooling Sessions 1–5. RT comes exclusively from raw decoding GDF Status onsets at 512 Hz. Responses must occur after the stimulus and before the following fixation (recording end for the last trial); missing or ambiguous responses stay NaN. Practice and uncertain alignments are excluded, with raw behavioral fingerprints verifying consolidated run identity.

Within-run consecutive pairs use `decoding_bci_output` (1 positive, 0 negative, 3 ambivalent), not `classification_output`. Correct responses, finite positive RT, and RT ≥150 ms precede one-pass ±3 sample-SD trimming within participant × session × current trial type, pooling runs and preceding feedback. Participants losing >50% of valid candidate pairs are excluded (exactly 50% allowed). Technical losses are reported separately. Control references use trials 2–60, the same cleaning and subject rule, and no feedback categories.

Participant means provide equal participant weighting. Four repeated-measures ANOVAs use model-specific complete cells, with Mauchly tests and Greenhouse–Geisser correction when violated. Paired t-tests use pairwise-complete subjects and Holm correction within each set of three contrasts; difference CIs are unadjusted. Session and run are not ANOVA factors. This exploratory analysis estimates association, not causation.


In [ ]:
from posthoc_analysis.feedback_rt import (
    load_feedback_rt_input, reconstruct_and_align_feedback_rt,
    build_feedback_rt_pairs, clean_feedback_rt_pairs, summarize_feedback_rt_quality,
)
from posthoc_analysis.feedback_rt_stats import (
    summarize_feedback_rt, run_feedback_rt_statistics,
    plot_feedback_next_trial_rt, save_feedback_rt_tables, validate_feedback_rt_results,
)
from IPython.display import display

feedback_rt_output = project_root / "analyses" / "feedback_next_trial_rt"
feedback_rt_figure_path = project_root / "figures" / "bci_previous_feedback_next_trial_rt.pdf"


### GDF reconstruction and alignment
The earlier comprehensive validation section audits training text/analysis files. This step independently validates every decoding GDF and its behavioral counterpart. A verified cache avoids repeating raw-file reads; changed input content or source-file metadata invalidates it. Use `force=True` to explicitly rebuild. Failed files remain logged and are never filled from text timing.


In [ ]:
feedback_bci_input, feedback_bci_source, feedback_input_audit = load_feedback_rt_input()
feedback_reconstruction = reconstruct_and_align_feedback_rt(
    feedback_bci_input, cache_dir=feedback_rt_output,
)
feedback_pairs, feedback_technical_pairs = build_feedback_rt_pairs(
    feedback_reconstruction["aligned_trials"]
)
feedback_quality = summarize_feedback_rt_quality(feedback_reconstruction, feedback_technical_pairs)
display(feedback_quality["technical_run_summary"])


### Sequential exclusions and participant eligibility
Denominators are validly aligned candidates. Summaries show exclusions before and after the >50% participant rule; participants with zero candidates are flagged separately. Previous-trial behavioral correctness is not an eligibility requirement.


In [ ]:
feedback_roster = feedback_bci_input[["subject_id", "group"]].drop_duplicates()
feedback_cleaning = clean_feedback_rt_pairs(feedback_pairs, roster=feedback_roster)
display(feedback_cleaning["subject_exclusions"])
display(feedback_cleaning["exclusion_summary"])


### BCI participant and group descriptions
Overall means pool eligible trials across both current trial types before averaging within each participant. Group uncertainty uses participant means and t-distribution 95% CIs. Absent cells remain NaN.


In [ ]:
feedback_descriptions = summarize_feedback_rt(
    feedback_cleaning["retained_trials"], feedback_cleaning["subject_exclusions"]
)
display(feedback_descriptions["bci_subject_summary"])
display(feedback_descriptions["bci_group_summary"])


### Repeated-measures ANOVAs and paired post hoc comparisons
Use `p_reported` with `df1_reported`/`df2_reported`, and `p_holm` for post hoc claims. Sphericity is tested on three feedback means; the interaction uses three present-minus-absent differences. The trial-type effect has two levels and needs no sphericity correction. Each model's included/omitted participant IDs are retained in `anova_membership`.


In [ ]:
feedback_statistics = run_feedback_rt_statistics(feedback_descriptions["bci_subject_summary"])
display(feedback_statistics["anova_results"])
display(feedback_statistics["anova_membership"])
display(feedback_statistics["posthoc_comparisons"])


### Mental-rehearsal descriptive reference
These unconditional participant means receive no feedback-dependent tests or inferential group comparisons.


In [ ]:
display(feedback_descriptions["control_subject_summary"])
display(feedback_descriptions["control_group_summary"])


### Manuscript figure and saved outputs
Small points and thin connecting lines show BCI participant means; large points/error bars show mean ± SEM across participants. Gray dashed lines show unconditional mental-rehearsal means. The same y-axis scale is used for all panels; BCI n is listed in Positive/Negative/Ambivalent order. Arial follows the specific figure request. Save vector PDF and 350-dpi PNG preview. The explicitly requested statistical tables are retained alongside the reconstruction/QC outputs in `analyses/feedback_next_trial_rt/`.


In [ ]:
feedback_numerical_validation = validate_feedback_rt_results(
    feedback_pairs, feedback_cleaning, feedback_descriptions, feedback_statistics
)
feedback_rt_figure = plot_feedback_next_trial_rt(feedback_descriptions, feedback_rt_figure_path)
feedback_exclusion_reports = {
    "bci_subject_exclusion_report": feedback_cleaning["subject_exclusions"].query("group == 'experimental'"),
    "control_subject_exclusion_report": feedback_cleaning["subject_exclusions"].query("group == 'control'"),
}
feedback_anova_tables = {
    "anova_" + model: feedback_statistics["anova_results"].query("model == @model")
    for model in ["overall", "feedback_x_trial_type", "distractor_present", "distractor_absent"]
}
save_feedback_rt_tables(
    feedback_rt_output, {"candidate_pairs": feedback_pairs}, feedback_quality,
    feedback_cleaning, feedback_exclusion_reports, feedback_descriptions,
    feedback_statistics, feedback_anova_tables,
    {"numerical_validation": feedback_numerical_validation},
)
feedback_rt_figure


## Pd pre/post waveforms and positive AUC after behavioral exclusions

The current Pd analysis applies behavioral trial exclusions; obsolete unfiltered workflows have been removed. Reuse `corrected_training_trials`: remove incorrect responses and timeouts, missing RTs, and RT <150 ms; then apply the existing single-pass participant/session mean ±3 sample-SD filter pooling every non-practice training run and all stimulus conditions. Exactly 150 ms and exactly ±3 SD are retained.

Rebuild stimulus-locked PO7/PO8 Pd from the retained distractor trials. **Pool all retained distractor trials with equal trial weighting within participant/session**, then average participants equally. Runs and distractor sides are weighted by their retained trial counts. Keep the canonical 0.1–20 Hz filter, −0.2 to <0 s baseline, and existing amplitude scaling. Plot light/dark orange for BCI and light/dark blue for mental rehearsal at the existing poster size. Positive AUC is the integral of max(participant/session Pd, 0) from 0.2–0.5 s; it is not the average of single-trial AUCs.

Retain the existing Group × Session mixed ANOVA and four planned contrasts, including their unadjusted P values, effect sizes, and pointwise 95% CIs. Expensive rebuilt waveforms and QC are cached in `analyses/eeg_pd/behavior_filtered/`; the cache checks trial content, raw-file metadata, and analysis code before reuse. The documented missing e30 Session-1 run remains reported; available runs contribute their retained trials.

In [ ]:
from posthoc_analysis.consolidated import validate_all_files_comprehensive
from posthoc_analysis.pd_filtered import (
    rebuild_behavior_filtered_pd,
    analyze_and_plot_behavior_filtered_pd,
)

In [ ]:
# Reports all training text/analysis issues, including the documented e30 missing run.
# The rebuild additionally requires strict raw/behavioral alignment for every loaded run.
pd_training_files_valid = validate_all_files_comprehensive()
pd_filtered_results = rebuild_behavior_filtered_pd(
    corrected_training_trials,
    cache_dir=project_root / "analyses" / "eeg_pd" / "behavior_filtered",
)

In [ ]:
display(pd_filtered_results["exclusion_summary"])
display(pd_filtered_results["rt_bounds"])
display(pd_filtered_results["run_trial_counts"])
print("Retained distractor trials by group/session:")
display(pd_filtered_results["trials"].groupby(["group", "session_id"])
        ["retained_for_pd"].sum().rename("n_retained_distractor_trials").to_frame())

In [ ]:
pd_filtered_auc_results = analyze_and_plot_behavior_filtered_pd(
    pd_filtered_results, output_dir=project_root / "figures",
)
display(pd_filtered_auc_results["cell_summary"])
display(pd_filtered_auc_results["anova_table"])
display(pd_filtered_auc_results["planned_contrasts"])
pd_filtered_auc_results["anova_diagnostics"]

In [ ]:
for group, figure in pd_filtered_auc_results["figures"].items():
    display(figure)
print("Saved figures:")
for path in pd_filtered_auc_results["saved_paths"]:
    print(path)

## Filtered Pd figures with mean distractor RT reference lines

Add vertical dashed lines at the Session-1 and Session-5 mean RTs, using each waveform's orange (BCI) or blue (mental-rehearsal) hue. Reuse the exact retained distractor trials from `pd_filtered_results`, all with Status-derived RT. Average trials within participant/session, then average participant means equally, matching the behavioral distractor-RT summary. The waveform data and positive-AUC statistics do not change. PDF and PNG exports replace the corresponding filtered Pd figures.

In [ ]:
from posthoc_analysis.pd_rt_plot import plot_behavior_filtered_pd_with_mean_rt

In [ ]:
pd_filtered_rt_figures = plot_behavior_filtered_pd_with_mean_rt(
    pd_filtered_results, output_dir=project_root / "figures",
)
display(pd_filtered_rt_figures["rt_summary"])
for figure in pd_filtered_rt_figures["figures"].values():
    display(figure)
pd_filtered_rt_figures["saved_paths"]

## Exploratory positive Pd AUC: 350–450 ms

Repeat the positive-AUC analysis in the alternative 350–450 ms window, reusing `pd_filtered_results` and exactly the same retained trials, participant/session waveforms, and equal participant weighting. Compute the integral of max(Pd, 0) on the participant/session mean waveform, then run the same Group × Session mixed ANOVA and four contrasts. P values and pointwise 95% CIs remain unadjusted, including across the two windows; treat this additional window analysis as exploratory.

Match the original integration convention: use recorded samples inside the requested window, without boundary interpolation. At 512 Hz, 51 samples span 351.5625–449.21875 ms. Store results separately in `pd_auc_350_450_results`; the 200–500 ms analysis and figures remain available above.

In [ ]:
from posthoc_analysis.pd_window_stats import analyze_filtered_pd_auc_window

In [ ]:
pd_auc_350_450_results = analyze_filtered_pd_auc_window(
    pd_filtered_results, time_window=(0.350, 0.450),
)
display(pd_auc_350_450_results["cell_summary"])
display(pd_auc_350_450_results["anova_table"])
display(pd_auc_350_450_results["planned_contrasts"])
pd_auc_350_450_results["anova_diagnostics"]

## Online ROC AUC: constrained label-shuffle reference

Estimate a reference distribution for the existing **BCI AUC Across Sessions** plot using `bci_auc_results['dataframe']`. Keep the recorded posterior scores fixed; draw 10,000 independent legal label orders within each participant/session/run (30 distractor, 30 absent; maximum three consecutive identical labels). Pool runs before computing each participant/session ROC AUC, then average participants equally within each group/session. Include ambivalent trials and retain the original missing-label exclusions. Use seed 20261103.

**Interpretation:** This is a fixed-score label-shuffle benchmark. Source inspection confirmed that the known stimulus selects decoder R, L, or N before one posterior is saved. These shuffles break that routing relationship; their reference tail probabilities do **not** establish above-chance EEG discrimination by the full online system, population generalization, or training improvement. A routing-preserving null would require applying each relevant saved decoder to every trial, then selecting the decoder under each shuffled condition. The current consolidated CSV contains only the selected score. See `doc/bci_auc_chance.md` for the audit and source links.

The central 95% reference interval and one-sided 95th percentile are distinct. Observed error bars remain SEM across participants. `p_label_shuffle` is the one-sided Monte Carlo reference tail probability with a +1 correction; Holm adjustment covers all 10 group/session points. Participant reference intervals are descriptive; no participant significance tests are reported.


In [ ]:
from posthoc_analysis.bci_auc_chance import (
    compute_bci_auc_shuffle_reference,
    plot_bci_auc_shuffle_reference,
)


In [ ]:
bci_auc_shuffle_results = compute_bci_auc_shuffle_reference(
    bci_auc_results['dataframe'],
    n_permutations=10_000,
    random_seed=20261103,
    alpha=0.05,
)


In [ ]:
display(bci_auc_shuffle_results['group_summary'])
display(bci_auc_shuffle_results['excluded_runs'])
bci_auc_shuffle_results['settings']


### AUC with the conditional label-shuffle reference

Observed group means ± SEM (16 participants per group) retain the original AUC calculation. Gray shading is the pointwise central 95% label-shuffle range; the dotted line is its one-sided 95th percentile. The dashed line at 0.5 denotes label-independent ranking. These are conditional reference bounds, **not** confidence intervals on observed group means or routing-preserving chance bounds. Paper typography; PDF and 300-dpi PNG are saved separately from the original figure.


In [ ]:
bci_auc_shuffle_plot = plot_bci_auc_shuffle_reference(
    bci_auc_shuffle_results,
    save=True,
)


## Combined AUC over sessions with mean shuffled AUC and range

Match the attached paper figure `figures/for paper/bci_auc_across_sessions.pdf`: exact exported page size **255.4946095 × 228.5775 points** (3.548536 × 3.174688 inches), axes at (54.848513, 32.35625)–(155.442112, 208.09625) points, original Arial typography, title, axis limits, and right-side group legend. Export the measured page without automatic cropping so the PDF dimensions remain exact.

Overlay the **actual mean shuffled AUC** and the pointwise **central 95% shuffle range** (2.5th–97.5th percentiles). Observed points and bars retain group mean ± SEM. Color-matched dashed/dotted reference lines show each group's permutation mean; light bands show its shuffle range. A compact reference key sits below the original group legend. No separate upper-percentile cutoff or theoretical 0.5 line is drawn.

These fixed-score reference bounds do not preserve condition-dependent decoder selection and are not observed-mean confidence intervals. This separate plot cell reuses `bci_auc_shuffle_results` without rerunning permutations and saves its PDF/PNG under the existing combined-reference filename.


In [ ]:
import importlib
import posthoc_analysis.bci_auc_chance as bci_auc_chance
importlib.reload(bci_auc_chance)
from posthoc_analysis.bci_auc_chance import plot_bci_auc_shuffle_reference


In [ ]:
bci_auc_combined_shuffle_plot = plot_bci_auc_shuffle_reference(
    bci_auc_shuffle_results,
    layout="combined",
    save=True,
)


## Pd positive AUC across pre, five training sessions, and post

Plot 200–500 ms positive AUC with separate BCI and mental-rehearsal lines (participant mean ± SEM; poster typography). Reuse the exact behavior-filtered pre/post waveforms in `pd_filtered_results`. For the five training points, use **all distractor trials from the decoding runs**, without accuracy, RT, BCI-feedback, or artifact exclusions. Omit explicitly named practice folders.

Compute **contra − ipsi before averaging or rectifying**: PO7−PO8 for right distractors, PO8−PO7 for left distractors. Pool trials equally within participant/time point, then integrate `max(mean Pd, 0)` using recorded samples in 200–500 ms. Average participant AUCs equally within each group. Retain the existing 0.1–20 Hz filter, −0.2 to <0 s baseline, and amplitude scaling.

Explicitly approved exceptions: keep all five e27 Training-1 runs and all seven e22/e37 Training-5 runs; use e52 Training-3's available EEG and audit its one missing GDF. Other malformed files cause a visible validation failure. Run counts and preprocessing warnings are retained with the expensive waveform cache. Pre/post and training have different trial-inclusion policies; their connecting lines are descriptive, with no statistical test added.


In [ ]:
import importlib
import posthoc_analysis.pd_longitudinal as pd_longitudinal
importlib.reload(pd_longitudinal)

from posthoc_analysis.pd_longitudinal import (
    build_seven_point_pd,
    compute_seven_point_pd_auc,
    plot_seven_point_pd_auc,
)


In [ ]:
# Reuse pd_filtered_results and the comprehensive training validation above.
# Validate decoding text/EEG alignment separately; reuse provenance-checked caches.
pd_seven_results = build_seven_point_pd(
    pd_filtered_results,
    cache_dir=project_root / "analyses" / "eeg_pd" / "seven_point",
    allow_available_runs=True,  # Explicitly approved extra runs and missing-EEG audit.
    workers=4,
)


In [ ]:
# Small reusable participant/session table for downstream inspection.
pd_seven_auc = compute_seven_point_pd_auc(pd_seven_results, time_window=(0.2, 0.5))
display(pd_seven_auc["group_summary"])
display(pd_seven_auc["participant_auc"][[
    "subject_id", "group", "point_label", "positive_auc_uv_sec",
    "n_distractor_trials", "n_runs", "trial_policy",
]])
display(pd_seven_results["run_trial_counts"].query("status == 'missing_eeg'"))


In [ ]:
pd_seven_figure = plot_seven_point_pd_auc(
    pd_seven_auc, output_dir=project_root / "figures", style="poster",
)
display(pd_seven_figure["figure"])
pd_seven_figure["saved_paths"]


## No-distractor target locations in pre/post training

Count lateral (position codes 2/4, right/left) versus midline (1/3, up/down) targets in the full non-practice training table. Reuse `corrected_training_trials` before accuracy, RT, or EEG filtering. Validate full 60-trial runs and show available-run counts; e30 Session 1 has seven runs.


In [ ]:
from posthoc_analysis.target_positions import summarize_no_distractor_target_positions


In [ ]:
no_distractor_target_counts = summarize_no_distractor_target_positions(
    corrected_training_trials
)
display(no_distractor_target_counts["overall"])
display(no_distractor_target_counts["by_session"])
display(no_distractor_target_counts["by_position"])


## Pre-task sensor waveforms for the right and left decoder class comparisons

Use **all** available Session 1 non-practice training trials, without accuracy, RT, artifact, class-balancing, or pruning exclusions. Average trials equally within participant, then average participants equally across both groups (mean ± SEM). Right panel: right distractor versus all no-distractor trials, **PO7 − PO8**. Left panel: left distractor versus the same no-distractor trials, **PO8 − PO7**. All no-distractor target positions are included. These are sensor-level class comparisons before xDAWN; the requested right-panel sign is opposite the fixed right-minus-left decoder feature convention.

Reuse canonical 0.1–20 Hz zero-phase filtering, 512 Hz stimulus-locked epochs, and baseline samples 154:256 (−0.2 to <0 s). Use native amplitude scaling from the existing EEG pipeline. Shade 200–500 ms; no smoothing or statistical test. Poster typography. Raw source fingerprints, code hashes, counts, and warnings accompany compact participant caches. The documented seven-run e30 pre-session is retained.


In [ ]:
from posthoc_analysis.pre_decoder_waveforms import (
    build_pre_decoder_waveforms,
    plot_pre_decoder_waveforms,
)


In [ ]:
# Reuse cohort identity from the unfiltered training table and comprehensive validation above.
pre_decoder_conditions = build_pre_decoder_waveforms(
    subjects=sorted(corrected_training_trials.subject_id.unique()),
    cache_dir=project_root / "analyses" / "eeg_pd" / "pre_decoder_conditions",
    workers=4,
)
display(pre_decoder_conditions["run_audit"])


In [ ]:
pre_decoder_condition_plot = plot_pre_decoder_waveforms(
    pre_decoder_conditions, output_dir=project_root / "figures", style="poster",
)
display(pre_decoder_condition_plot["figure"])
display(pre_decoder_condition_plot["trial_counts"])
pre_decoder_condition_plot["saved_paths"]


## Posterior theta across pre, five intervention sessions, and post

Plot left posterior (PO7, P7, P5), right posterior (PO8, P8, P6), and bilateral six-electrode theta for right distractor, left distractor, pooled left/right distractor, no-distractor, and all trials. “Both distractors” means the two trial types pooled, not two simultaneous distractors.

Match the original midfrontal analysis: 4–8 Hz (0.5-Hz steps), 100–500 ms, −300 to −100 ms session-specific pooled power baseline, full 61-channel surface Laplacian, and the original Morlet schedule. Pre/post retain the original correct-response, RT ≥150 ms, single-pass participant/session ±3 sample-SD mask; intervention sessions retain all outcomes. Keep the original practice/missing-run exceptions and all 16 participants per group. Pool linear trial power before dB, then average time, frequency, and ROI electrodes. Group means ± SEM weight participants equally. Figures share y-axis limits and use the original paper typography. Descriptive plots only; no inferential statistics.

The explicit reference directory points to the existing validated midfrontal analysis. Override `PD_THETA_REFERENCE_ROOT` if it is moved. New side-specific power checkpoints are saved locally and reused only after source/mask/settings/code checks. The original analysis is preserved.

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display
from posthoc_analysis.posterior_theta import (
    prepare_posterior_theta_inputs,
    build_posterior_theta_cache,
    summarize_posterior_theta,
    plot_posterior_theta,
    verify_posterior_theta_intervention_reference,
)

posterior_theta_reference = Path(os.environ.get(
    'PD_THETA_REFERENCE_ROOT',
    str(project_root / 'exploratory_results' / 'theta_paper_reproduction'),
))
posterior_theta_cache = project_root / 'analyses' / 'eeg_theta' / 'posterior_seven_point'
posterior_theta_figures_dir = project_root / 'figures' / 'posterior_theta'


In [ ]:
posterior_theta_inventory, posterior_theta_mask, posterior_theta_settings = prepare_posterior_theta_inputs(
    posterior_theta_reference, posterior_theta_cache,
)
posterior_theta_reference_qc = verify_posterior_theta_intervention_reference(
    posterior_theta_reference, posterior_theta_cache,
)


### Extract side-specific posterior power

This cell performs the expensive raw-EEG step on the first run. Subsequent runs validate and reuse the checkpoints. Detailed run-level summaries are written to the cache log; missing or malformed inputs stop processing rather than silently changing inclusion.

In [ ]:
import contextlib

with (posterior_theta_cache / 'notebook_extraction.log').open('w') as posterior_theta_log:
    with contextlib.redirect_stdout(posterior_theta_log):
        posterior_theta_runs = build_posterior_theta_cache(
            posterior_theta_inventory, posterior_theta_mask, posterior_theta_settings,
            posterior_theta_cache, n_jobs=2,
        )
print(f"Validated {len(posterior_theta_runs):,} runs; {posterior_theta_runs.n_retained.sum():,} retained trials.")
print(f"Run log: {posterior_theta_cache / 'notebook_extraction.log'}")


In [ ]:
posterior_theta_participants, posterior_theta_qc = summarize_posterior_theta(posterior_theta_cache)
posterior_theta_paths, posterior_theta_group_summary = plot_posterior_theta(
    posterior_theta_participants, posterior_theta_figures_dir,
)
for posterior_theta_roi, posterior_theta_exports in posterior_theta_paths.items():
    print(posterior_theta_roi.replace('_', ' ').title())
    display(Image(filename=posterior_theta_exports['png'], width=900))


## Posterior theta: Group × Session × Trial type × EEG side ANOVA

Use the preceding validated posterior theta participant summaries. Group (BCI, mental rehearsal) is between participants; Session, Trial type (left distractor, right distractor, no distractor), and EEG side (left PO7/P7/P5, right PO8/P8/P6) are repeated factors. The DV is 4–8 Hz, 100–500 ms baseline-normalized Laplacian theta in dB. Exclude pooled/bilateral derived categories from these models; retain the same trials and all 32 participants.

The seven-point model is an exploratory overview. Separate five-session intervention and pre/post assessment models distinguish phases with different task contexts and trial-selection rules. Fit all 15 fixed effects per model, using a separate subject-within-group error stratum for each within-participant effect. Report Greenhouse–Geisser-adjusted degrees of freedom and p values for multi-df repeated effects and their Group interactions; one-df effects require no correction. Retain raw p values and partial eta squared. GG addresses sphericity, **not multiplicity**: these exploratory omnibus tests are not adjusted across effects or the three overlapping models.

Each participant and each trial-type/hemisphere cell receives equal weight for marginal effects. Within cells, the theta values preserve trial-weighted linear-power pooling before dB. Thus, the three-level Trial type marginal mean is not the plotted all-trial pooled-power value. Target-position differences remain: distractor-present trials have midline targets, while no-distractor trials include lateral targets. No target-position sensitivity analysis or post hoc tests are included in this step.

Numerical QA independently reconstructs each numerator/error sum of squares, F statistic and epsilon in the original cell space, checks the total SS partition, and compares one-df effects with OLS t² tests.

In [ ]:
from posthoc_analysis.posterior_theta_stats import (
    prepare_posterior_theta_anova,
    run_posterior_theta_mixed_anova,
    save_posterior_theta_anova_audit,
)

posterior_theta_anova_input = prepare_posterior_theta_anova(posterior_theta_participants)
posterior_theta_anova_columns = [
    'effect', 'F', 'df_num_GG', 'df_den_GG', 'p_GG',
    'p_uncorrected', 'epsilon_GG', 'partial_eta_squared',
]


### Seven-point overview

Pre, intervention sessions 1–5, post: 1,344 repeated cells, 42 per participant. Interpret Session effects in light of the assessment/intervention phase differences.

In [ ]:
posterior_theta_anova_seven = run_posterior_theta_mixed_anova(
    posterior_theta_anova_input, model='seven_point',
)
display(posterior_theta_anova_seven['anova'][posterior_theta_anova_columns])


### Intervention sessions 1–5

Five-session model: 960 repeated cells, 30 per participant. All available intervention trial outcomes remain included.

In [ ]:
posterior_theta_anova_intervention = run_posterior_theta_mixed_anova(
    posterior_theta_anova_input, model='intervention',
)
display(posterior_theta_anova_intervention['anova'][posterior_theta_anova_columns])


### Pre/post assessment

Assessment model: 384 repeated cells, 12 per participant. Both assessments use the original behavioral/RT eligibility mask.

In [ ]:
posterior_theta_anova_assessment = run_posterior_theta_mixed_anova(
    posterior_theta_anova_input, model='pre_post',
)
display(posterior_theta_anova_assessment['anova'][posterior_theta_anova_columns])


In [ ]:
posterior_theta_anova_results = {
    'seven_point': posterior_theta_anova_seven,
    'intervention': posterior_theta_anova_intervention,
    'pre_post': posterior_theta_anova_assessment,
}
posterior_theta_anova_qc = save_posterior_theta_anova_audit(
    posterior_theta_anova_results,
    posterior_theta_cache / 'anova_validation.json',
)


The Group × Session and Group × Session × Trial type × EEG side effects are not significant in any model. Group × Trial type is significant at the nominal 0.05 level in the seven-point and intervention models, but not in pre/post assessments. This concerns average trial-type differences between groups; it does not establish different changes over sessions. All results remain exploratory and unadjusted for multiplicity across omnibus tests/models. Full numerical results, error strata, descriptives, exact input data, and validation records are available in the result dictionaries above.

## Posterior theta: Group × distractor presence × intervention session

Mixed ANOVA: Group (BCI versus mental rehearsal) × Condition (distractor present versus absent) × Session (1–5), matching the five intervention levels in the midfrontal ANOVA. The dependent variable is normalized 4–8 Hz theta, 100–500 ms, averaged over PO7/P7/P5/PO8/P8/P6. Reuse the validated posterior summaries above. Distractor-present pools left/right trial **linear power before dB**, using `both_distractors`; it does not average the separate condition dB values. All intervention outcomes remain included (32 participants, 16/group; 69,000 trials).

Greenhouse–Geisser corrections apply to effects involving Session; uncorrected results are retained. These exploratory tests have no correction across the seven omnibus effects or previous models. Target-position distributions differ between distractor-present and absent trials. The three-way interaction tests whether the between-group difference in the condition effect changes across sessions.

In [ ]:
from posthoc_analysis.posterior_theta_stats import run_posterior_theta_condition_anova

In [ ]:
posterior_theta_condition_anova = run_posterior_theta_condition_anova(posterior_theta_participants)
display(posterior_theta_condition_anova['anova'][[
    'effect', 'F', 'df_num_GG', 'df_den_GG', 'p_GG',
    'p_uncorrected', 'epsilon_GG', 'partial_eta_squared',
]])

In [ ]:
# Equal-session participant means; descriptive condition contrasts, not extra tests.
display(posterior_theta_condition_anova['condition_means'])
display(posterior_theta_condition_anova['descriptives'])
display(posterior_theta_condition_anova['validation'])

## Posterior theta: exploratory Group × Condition follow-ups

Match the latest midfrontal follow-up: average the five intervention session estimates equally within each participant and condition, using all six posterior electrodes. Run four two-sided simple effects: paired present-versus-absent tests within BCI and mental rehearsal; Welch between-group tests separately for present and absent conditions. Report **unadjusted exploratory p-values**, marginal 95% confidence intervals, paired Cohen's dz and between-group pooled-SD Cohen's d. No session-specific tests or new exclusions.

The between-group difference in present-minus-absent contrasts reconstructs the existing Group × Condition interaction (pooled-variance t² = F). This is the same omnibus hypothesis, not a fifth new follow-up test. These averages do not test changes over sessions; the earlier three-way interaction remains the relevant test of that question. Condition-related target-position differences remain part of these data.

In [ ]:
from posthoc_analysis.posterior_theta_posthoc import run_posterior_theta_group_condition_followups

In [ ]:
posterior_theta_condition_followups = run_posterior_theta_group_condition_followups(
    posterior_theta_condition_anova['input'],
    posterior_theta_condition_anova['anova'],
    correction=None,
)

In [ ]:
display(posterior_theta_condition_followups['descriptives'])
display(posterior_theta_condition_followups['simple_effects'])
display(posterior_theta_condition_followups['interaction_contrast'])
display(posterior_theta_condition_followups['validation'])

## Posterior theta: separate distractor and no-distractor seven-point figures

Match the midfrontal trajectory style for the combined PO7/P7/P5/PO8/P8/P6 ROI. Export separate vector PDFs (plus SVG and PNG) for pooled distractor-present and distractor-absent trials. Points/lines show equal-participant means; error bars show between-participant SEM (16 participants/group). Both figures use common y limits. Pre/post retain the original behavioral screen; intervention sessions include all outcomes. Dashed separators mark phase boundaries. No new statistical tests or exclusions; reuse the validated posterior summaries above.

In [ ]:
from posthoc_analysis.posterior_theta import plot_posterior_theta_condition_trajectories
from IPython.display import Image, display

In [ ]:
posterior_theta_condition_figures, posterior_theta_condition_plot_summary = plot_posterior_theta_condition_trajectories(
    posterior_theta_participants, project_root / 'figures' / 'posterior_theta',
)

In [ ]:
for condition, paths in posterior_theta_condition_figures['paths'].items():
    display(Image(filename=paths['png']))
    print(condition, 'PDF:', paths['pdf'])

# Consolidated midfrontal theta, Stroop theta, and intervention behavior

Retained analyses 3 and 20–24 from the former worktree. Alpha, broad mechanism screening, and superseded theta variants (25–27) are excluded. Existing numerical settings, trial masks, and correction choices are preserved.


## 1. Paths and imports

Run from the repository root or the `notebooks/` directory. Set `STUDY_ROOT` below if the source-data directory differs from the repository default. `PD_STUDY_ROOT` and `PD_THETA_OUTPUT` environment variables can also override these paths. The notebook uses the existing standard_1020 template to reproduce the PDF; it does not substitute the subsequently supplied channel-location file.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'code' / 'src' / 'posthoc_analysis').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from this repository or its notebooks directory.')
sys.path.insert(0, str(REPO_ROOT / 'code' / 'src'))

from posthoc_analysis.config import PROJECT_ROOT
STUDY_ROOT = Path(os.environ.get('PD_STUDY_ROOT', str(PROJECT_ROOT))).expanduser()
OUTPUT_ROOT = Path(os.environ.get('PD_THETA_OUTPUT', str(REPO_ROOT / 'exploratory_results' / 'theta_paper_reproduction'))).expanduser()
CACHE_DIR = OUTPUT_ROOT / 'theta_cache'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
OPTIONAL_CACHE = None  # Retained local cache is sufficient; obsolete copies were removed.
N_JOBS = 2

from IPython.display import Image, display
from posthoc_analysis.theta_paper_reproduction import (
    prepare_theta_paper_inputs, build_theta_paper_cache, summarize_theta_paper_cache,
    seal_theta_paper_outputs, validate_theta_paper_results, verify_theta_paper_raw_checkpoint,
)
from posthoc_analysis.theta_roi_anova import prepare_theta_roi_change, run_theta_roi_mixed_anova
from posthoc_analysis.theta_session_overview import plot_pooled_laplacian_theta_sessions
from posthoc_analysis.theta_window_power import summarize_theta_roi_window_power, plot_theta_roi_window_bars
from posthoc_analysis.theta_rt_association import analyze_theta_rt_changes, plot_theta_rt_changes

print('Study:', STUDY_ROOT)
print('Outputs:', OUTPUT_ROOT)
print('Optional reuse cache:', OPTIONAL_CACHE)


## 2. Raw-data validation and RT cleaning

Read Status-channel events at 512 Hz and validate each run against its paired task files. Retain correct trials with RT ≥150 ms, then apply one-pass ±3 sample SD within participant/session, pooling conditions and runs. Missing responses remain missing and are excluded. The implementation independently checks the mask against the established behavioral functions. Expected totals: 383 runs, 22,980 trials, 47 missing responses, 21,940 retained trials. The missing-response exclusion count is 24 after incorrect/timeout trials are removed; the other 23 missing responses are already excluded as incorrect/timeout.


In [ ]:
inventory, trial_mask, theta_settings = prepare_theta_paper_inputs(STUDY_ROOT, OUTPUT_ROOT)
display(trial_mask.groupby('exclusion').size().rename('trials').to_frame())


## 3. Single-trial Laplacian theta power (resumable)

Continuous zero-phase FIR filtering at 0.1–20 Hz → spherical-spline surface Laplacian across 61 recorded scalp channels (λ=10⁻⁵, stiffness 4, 50 Legendre terms; template sphere) → epochs −1 to +1.5 s → single-trial zero-mean Morlet power at 4:0.5:8 Hz. Cycles follow the logarithmic 3-to-10 schedule over 2–60 Hz (approximately 3.83–4.90 within theta). No voltage-baseline subtraction, amplitude rejection, ICA or additional channel repair.

The baseline is retained-trial mean linear power over −300 to −100 ms, pooled across conditions per participant/session/channel/frequency. Normalize mean linear power as `10*log10(power/baseline)`, then average frequencies and 100–500 ms. Both-trial summaries pool linear power before normalization; they are not the mean of the two condition-specific dB estimates.


In [ ]:
run_log = build_theta_paper_cache(
    inventory, trial_mask, theta_settings, CACHE_DIR,
    reuse_cache_dir=OPTIONAL_CACHE, n_jobs=N_JOBS,
)
theta_table = summarize_theta_paper_cache(CACHE_DIR)


### Raw-computation validation

Recompute one complete retained-trial run without a cache and verify that its linear baseline and stimulus power match the checkpoint. This exercises the same fallback used when no optional cache is present.


In [ ]:
raw_validation = verify_theta_paper_raw_checkpoint(
    inventory, trial_mask, theta_settings, CACHE_DIR, OUTPUT_ROOT / 'validation',
)


## 4. Group × distractor-presence mixed ANOVA

DV: Session 5 minus Session 1 normalized theta, averaging Fz/FCz/Cz equally after channel-specific normalization, separately for distractor and no-distractor trials. Group is between participants; distractor presence is within participants. All 32 participants remain. Report the PDF’s unadjusted two-sided omnibus p values, partial η², and group contrast CI. The group means here equally average condition-specific changes; this differs slightly from the pooled-power summaries below.


In [ ]:
roi_changes = prepare_theta_roi_change(CACHE_DIR, OUTPUT_ROOT / 'anova', roi=('FZ', 'FCZ', 'CZ'))
anova_results = run_theta_roi_mixed_anova(
    roi_changes, OUTPUT_ROOT / 'anova', correction=None, include_diagnostics=False,
)
display(anova_results['anova'][['effect', 'F', 'df_num', 'df_den', 'p_value', 'partial_eta_sq',
                              'estimate_db', 'ci95_low', 'ci95_high']])
display(roi_changes.groupby('group').change_db.mean().rename('condition_equal_mean_change_db').to_frame())


## 5. Figure 1: pooled pre-intervention, post-intervention and post–pre topographies

BCI and mental-rehearsal rows. Panel **a** contains pre/post maps; panel **b** contains paired post–pre changes, separated by extra space and a thin gray divider. Show names and dots only at Fz, FCz, Cz and bilateral P3/P4, P5/P6, P7/P8, PO3/PO4, PO5/PO6, PO7/PO8 ; hide all other electrode dots. All 61 channels still contribute to the maps. Preserve regularized spherical display interpolation and the common ±1.5 dB scale; display choices do not enter ROI statistics.


In [ ]:
topography = plot_pooled_laplacian_theta_sessions(
    CACHE_DIR, FIGURE_DIR / 'topography',
    electrode_labels=['Fz', 'FCz', 'Cz', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8',
                      'PO3', 'PO4', 'PO5', 'PO6', 'PO7', 'PO8'],
    group_panels=True, session_term='intervention', selected_electrodes_only=True,
)
display(Image(filename=topography['paths']['png']))

## 6. Baseline/stimulus power table and Figure 2

Pool retained trials across distractor conditions, average 4–8 Hz and Fz/FCz/Cz within each participant, then equally weight participants. Unnormalized wavelet power is displayed in 10⁸ arbitrary units (not µV² or spectral-density units). The third measure is the existing participant-normalized dB value. Bars show mean ± SEM; points and connecting lines show paired participants. No additional tests of unnormalized power are performed.


In [ ]:
window_participants, window_summary, window_validation = summarize_theta_roi_window_power(
    CACHE_DIR, OUTPUT_ROOT / 'window_power', roi=('FZ', 'FCZ', 'CZ'),
)
window_manifest = seal_theta_paper_outputs(OUTPUT_ROOT / 'window_power')
paper_power_table = window_summary[['group', 'session_id', 'baseline_power', 'stimulus_power', 'normalized_theta_db']].copy()
paper_power_table[['baseline_power', 'stimulus_power']] /= 1e8
paper_power_table = paper_power_table.rename(columns={'baseline_power': 'baseline_power_1e8_au',
    'stimulus_power': 'stimulus_power_1e8_au'})
display(paper_power_table.round(3))


In [ ]:
window_figure = plot_theta_roi_window_bars(window_participants, FIGURE_DIR / 'window_power', include_normalized=True)
display(Image(filename=window_figure['paths']['png']))


## 7. Theta–RT correlations and Figure 3

Theta change is post minus pre. **RT improvement is pre minus post** (Session 1 minus Session 5), so positive values indicate faster responses after training and match the manuscript’s primary theta–RT paragraph. Both outcomes pool the identical retained distractor and no-distractor trials. Pearson/Spearman correlations, confidence intervals, regression slopes, and plots all use RT improvement. The compatibility column `rt_change_ms` retains post-minus-pre RT but is not the tested outcome.

P values are two-sided, unadjusted and exploratory; CIs are pointwise Fisher-z intervals. The pooled correlation includes between-group separation and is not evidence of intervention-independent coupling. No mediation or causal inference is made.


In [ ]:
rt_changes, correlations, association_validation = analyze_theta_rt_changes(
    OUTPUT_ROOT / 'window_power', OUTPUT_ROOT / 'rt', CACHE_DIR,
    OUTPUT_ROOT / 'theta_rt_association', paper_only=True,
)
display(correlations[['scope', 'n', 'pearson_r', 'p_value', 'ci95_low', 'ci95_high', 'spearman_rho', 'spearman_p']])


In [ ]:
correlation_figure = plot_theta_rt_changes(rt_changes, correlations, FIGURE_DIR / 'theta_rt_association')
display(Image(filename=correlation_figure['png']))


## 8. Verify reproduction of the supplied PDF

The constants used by this check are rounded reference values from the PDF, not inputs to any calculation. A mismatch raises an error rather than silently reporting a reproduction. Numerical agreement is assessed at the precision printed in the PDF; exact raster appearance can depend on font/library versions.

The current manuscript, saved correlations and Figure 3 use pre-minus-post RT improvement. The reference validator checks the explicit direction metadata so current correlations agree directly with the PDF.


In [ ]:
pdf_checks = validate_theta_paper_results(
    anova_results['anova'], roi_changes, window_summary, correlations, OUTPUT_ROOT / 'validation',
)
display(pdf_checks)
print('Reproduction complete. Figures:', FIGURE_DIR)


In [ ]:
from posthoc_analysis.theta_rt_association import plot_theta_rt_changes


In [ ]:
theta_rt_pooled_figure = plot_theta_rt_changes(
    rt_changes, correlations, FIGURE_DIR / 'theta_rt_association', pooled_only=True,
)
display(Image(filename=theta_rt_pooled_figure['png']))


## 13. Stroop theta: congruent, incongruent and pooled topographies

Apply the same 61-channel Laplacian pipeline and 4–8 Hz wavelets as distractor theta, with a −300 to −100 ms power baseline and 100–300 ms target window. Show whole-scalp pre, post and post − pre maps for BCI and mental rehearsal, separately for congruent, incongruent and pooled trials. **Descriptive maps only; no ROI statistics or correlations yet.**

Use the current Stroop RT exclusions: incorrect responses, timeouts and one-pass participant/session mean ±3 sample SD, pooling both runs and both congruencies; **no minimum RT cutoff**. The same retained trials contribute to stimulus and baseline power. A common pooled-congruency baseline is used within each participant/session/channel/frequency, and overall conditions are pooled in linear power before dB normalization.

Stroop stimulus codes are 8 (congruent) and 32 (incongruent), checked against every behavioral row. Measured fixation is approximately 500 ms; the nominal baseline lies within fixation, although theta-wavelet temporal support extends into adjacent activity. Reuse the existing montage, reference, continuous filter and wavelet cycle schedule. See `doc/stroop_theta.md` for event-source evidence and validation details.

The revised target is 100–300 ms. Original 100–500 ms outputs remain in `OUTPUT_ROOT/stroop`; this window has a separate cache.

The earlier 300–700 ms results remain in `OUTPUT_ROOT/stroop_300_700ms`. The new window uses exact cached samples from the original 100–500 ms run.


In [ ]:
from posthoc_analysis.stroop_theta import (
    prepare_stroop_theta_inputs, rewindow_stroop_theta_cache,
    summarize_stroop_theta, plot_stroop_theta_topographies,
)

STROOP_THETA_ROOT = OUTPUT_ROOT / 'stroop_100_300ms'
STROOP_THETA_CACHE = STROOP_THETA_ROOT / 'cache'
STROOP_THETA_FIGURES = STROOP_THETA_ROOT / 'figures'

STROOP_TARGET_S = (0.1, 0.3)

STROOP_SOURCE_CACHE = OUTPUT_ROOT / 'stroop' / 'cache'


### Validate recordings and reproduce Stroop RT exclusions

Validate all 128 non-practice GDF/behavior pairs, condition/event alignment and 61-channel layout. This source scan does not compute EEG power. The current mask should match the behavioral workflow: 7,073 retained of 7,680 trials (438 timeouts, 57 incorrect, 112 SD outliers). Earlier per-run Stroop trimming in this worktree is superseded for this analysis.


In [ ]:
stroop_theta_inventory, stroop_theta_ledger, stroop_theta_settings = prepare_stroop_theta_inputs(
    STUDY_ROOT, CACHE_DIR, STROOP_THETA_CACHE, target_s=STROOP_TARGET_S,
)
display(stroop_theta_ledger.groupby(['group', 'session_id', 'exclusion_reason']).size().unstack(fill_value=0))


### Select the new window from validated power checkpoints

Validate source checksums, settings, trial masks and axes, then select 100–300 ms from the original 100–500 ms linear-power arrays. Baselines and trials remain identical.


In [ ]:
stroop_theta_run_log = rewindow_stroop_theta_cache(
    stroop_theta_inventory, stroop_theta_ledger, stroop_theta_settings,
    STROOP_THETA_CACHE, source_cache_dir=STROOP_SOURCE_CACHE,
)
display(stroop_theta_run_log[['subject_id', 'session_id', 'run_id', 'n_retained', 'n_target_samples']])


### Normalize and inspect retained condition counts

Pool both runs in linear power before normalization. Congruent, incongruent and pooled estimates share the participant/session baseline. Group maps weight participants equally; post − pre is computed within participants.


In [ ]:
stroop_theta_table, stroop_theta_counts = summarize_stroop_theta(STROOP_THETA_CACHE)
display(stroop_theta_counts.groupby(['group', 'session_id', 'condition']).n_trials.sum().unstack('condition'))


### Whole-scalp topoplots before ROI statistics

Each page shows BCI and mental-rehearsal rows with pre, post and post − pre columns. Use identical pre/post color limits across congruent, incongruent and pooled maps, and identical change limits across those conditions. Limits cover all rendered values without clipping. All electrodes are labeled; Fz/FCz/Cz can be inspected before subsequent ROI analysis.

A fourth page shows **incongruent − congruent** dB within each participant/session, followed by the post − pre change in this contrast. All six contrast panels share one symmetric color scale. This is a descriptive paired difference; no statistical test is performed.


In [ ]:
stroop_theta_figures = plot_stroop_theta_topographies(STROOP_THETA_CACHE, STROOP_THETA_FIGURES)
for condition in stroop_theta_figures['conditions']:
    display(Image(filename=stroop_theta_figures['figures'][condition]['paths']['png']))
print('Stroop theta PDF:', stroop_theta_figures['pdf'])


## 14. Midfrontal theta across pre, five intervention sessions, and post

Track Fz/FCz/Cz Laplacian theta (4–8 Hz, **100–500 ms**) across seven time points. Reuse the existing behavior-screened distractor pre/post values; retain **all trials** in experimental intervention (`decoding`) runs, including incorrect responses and timeouts. Normalize each intervention session to its own −300 to −100 ms baseline pooled across both conditions and all runs. Pool trial linear power before dB normalization; then average time, frequency and ROI electrodes.

Exclude practice runs using the documented folder exceptions (e22/e37 Session 5); retain all five e27 Session 1 experimental-labeled runs. The user approved using seven available runs for e52 Session 3 because Run 6 lacks its GDF. No other missing or invalid runs are silently omitted. Plot group means ± SEM with equal participant weights; no inferential statistics.


In [ ]:
from posthoc_analysis.theta_intervention import (
    prepare_intervention_theta_inputs, build_intervention_theta_cache,
    summarize_intervention_theta_trajectory, plot_intervention_theta_trajectory,
)

INTERVENTION_ROOT = OUTPUT_ROOT / 'intervention_trajectory'
INTERVENTION_CACHE = INTERVENTION_ROOT / 'cache'
INTERVENTION_FIGURES = INTERVENTION_ROOT / 'figures'
INTERVENTION_ROI = ('FZ', 'FCZ', 'CZ')
INTERVENTION_MISSING_GDF = [('e52', 3, 6)]  # Explicit user-approved exception.


### Validate intervention recordings and all-trial eligibility

No compatible intervention theta cache was found in the canonical study analyses or repository fallback. Existing assessment theta is reused. Validate each available EEG/analysis/trigger set and keep original run/trial identities, including the gap at e52 Session 3 Run 6.


In [ ]:
intervention_inventory, intervention_ledger, intervention_settings = prepare_intervention_theta_inputs(
    STUDY_ROOT, CACHE_DIR, INTERVENTION_CACHE,
    missing_gdf_keys=INTERVENTION_MISSING_GDF, roi=INTERVENTION_ROI,
)
display(intervention_inventory.groupby(['group', 'session_id']).agg(n_runs=('run_id', 'size'), n_trials=('n_trials', 'sum')))


### Compute session-wise theta power

Apply the inherited continuous filter and full 61-channel Laplacian, then compute Morlet power at the three midfrontal electrodes. Save resumable run checkpoints; no behavioral or EEG-amplitude exclusions are applied to intervention trials.


In [ ]:
intervention_run_log = build_intervention_theta_cache(
    intervention_inventory, intervention_ledger, intervention_settings,
    INTERVENTION_CACHE, n_jobs=N_JOBS,
)


### Assemble seven-session participant trajectories

For each session, distractor and no-distractor estimates share the same pooled baseline. The all-trial estimate pools linear power across conditions before normalization. Reuse the exact pre/post values, which retain their original behavioral exclusions.


In [ ]:
intervention_trajectories, intervention_summary_validation = summarize_intervention_theta_trajectory(
    INTERVENTION_CACHE, CACHE_DIR,
)
display(intervention_trajectories.groupby(['phase', 'session_label', 'condition']).agg(n_participants=('subject_id', 'nunique'), n_trials=('n_trials', 'sum')))

# Seal the regenerated local summary before downstream checksum validation.
seal_theta_paper_outputs(INTERVENTION_CACHE)


### Distractor, no-distractor and all-trial trajectories

The x axis is Pre, intervention sessions 1–5, and Post. The y axis is normalized theta power in dB. Each panel shows BCI (orange) and mental rehearsal (blue); error bars are between-participant SEM (n = 16 per group). Dashed separators mark the assessment/intervention boundaries, where eligibility policies differ.


In [ ]:
intervention_figure, intervention_group_summary = plot_intervention_theta_trajectory(
    intervention_trajectories, INTERVENTION_FIGURES,
)
display(Image(filename=intervention_figure['paths']['png']))
print('Seven-session theta PDF:', intervention_figure['paths']['pdf'])


## 15. Intervention theta: 2 × 2 × 5 mixed ANOVA

Test **Group** (BCI vs mental rehearsal; between participants) × **Distractor presence** (distractor vs no distractor; within participants) × **Session** (intervention sessions 1–5; within participants). Use the validated 4–8 Hz Fz/FCz/Cz values from Section 14, with the 100–500 ms target and −300 to −100 ms baseline. Pre/Post assessment and pooled-all-trial values are not included in this model.

All 32 participants have all 10 repeated cells. Every available intervention trial is retained; e52 Session 3 uses the seven available runs as approved. Participants receive equal weight. No additional EEG computation is needed.


In [ ]:
from posthoc_analysis.theta_intervention_anova import (
    prepare_intervention_theta_anova,
    run_intervention_theta_mixed_anova,
)

INTERVENTION_ANOVA_DIR = INTERVENTION_ROOT / 'anova_2x2x5_uncorrected'


### Validate and select the intervention values

Verify input checksums and analysis settings, then select the 320 participant × condition × session values from the existing trajectory cache. Save the exact ANOVA input and provenance.


In [ ]:
intervention_anova_input, intervention_anova_provenance = prepare_intervention_theta_anova(
    INTERVENTION_CACHE, INTERVENTION_ANOVA_DIR,
)
display(intervention_anova_input.groupby(['group', 'condition', 'session_id']).agg(
    participants=('subject_id', 'nunique'), trials=('n_trials', 'sum'),
))


### Omnibus effects and effect sizes — uncorrected

The balanced mixed ANOVA uses separate subject-within-group error strata. At the user's request, report **uncorrected degrees of freedom and p values, without Greenhouse–Geisser correction**. Session terms use df = (4, 120); the remaining terms use df = (1, 30). Report partial η² for every effect. No multiplicity adjustment is applied.

Every sum of squares and F statistic is checked independently using classical marginal-mean decompositions. The earlier GG-adjusted exports remain under `anova_2x2x5/`; current uncorrected outputs are saved separately under `anova_2x2x5_uncorrected/`.


In [ ]:
intervention_anova_results = run_intervention_theta_mixed_anova(
    intervention_anova_input, INTERVENTION_ANOVA_DIR, correction=None,
)
display(intervention_anova_results['anova'][[
    'effect', 'F', 'df_num', 'df_den', 'p_uncorrected', 'partial_eta_squared',
]].round(5))
display(intervention_anova_results['descriptives'])


### Results — uncorrected

The Group × Distractor presence interaction was significant, F(1, 30) = 8.37, p = .00703, partial η² = .218. Neither Group × Session, F(4, 120) = 0.21, p = .935, partial η² = .0068, nor the three-way interaction, F(4, 120) = 1.03, p = .396, partial η² = .033, was significant. Other p values were .092 (Group), .082 (Distractor presence), .101 (Session) and .0719 (Distractor presence × Session). All p values and degrees of freedom are unadjusted.


## 16. Exploratory follow-up: Group × Distractor presence

Decompose the significant interaction from Section 15 by averaging the five intervention-session dB values within each participant and condition. This uses equal session and participant weights, consistent with the omnibus interaction, with no additional exclusions. Parameters remain 4–8 Hz, Fz/FCz/Cz, 100–500 ms and −300 to −100 ms baseline.

Run four two-sided simple effects: paired distractor-versus-no-distractor tests within each group and Welch BCI-versus-control tests within each condition. At the user's request, report raw two-sided p values without Holm or other multiplicity correction. Report marginal (unadjusted) 95% confidence intervals, paired Cohen dz and between-group pooled-SD Cohen d. These standardized effect sizes have different denominators and should not be directly equated.

Also express the original interaction as (distractor − no distractor) in BCI minus that contrast in control. Its pooled-variance t² must reproduce the omnibus F; this is the same tested hypothesis, not a fifth new simple effect. No session-specific follow-up tests are run.


In [ ]:
from posthoc_analysis.theta_intervention_posthoc import (
    run_intervention_theta_group_condition_followups,
)

INTERVENTION_FOLLOWUP_DIR = INTERVENTION_ANOVA_DIR / 'group_condition_followups'


### Simple effects, descriptives and interaction contrast

Reuse the validated input and omnibus results from Section 15. The underlying function verifies all 320 cells, participant group assignments, five sessions per participant/condition, paired alignment, t/df/CI calculations and equivalence with the original interaction.

Methods: [paired t tests](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html), [Welch t tests](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html).


In [ ]:
intervention_group_condition_followups = run_intervention_theta_group_condition_followups(
    intervention_anova_input, intervention_anova_results['anova'], INTERVENTION_FOLLOWUP_DIR, correction=None,
)
display(intervention_group_condition_followups['descriptives'].round(5))
display(intervention_group_condition_followups['simple_effects'].round(5))
display(intervention_group_condition_followups['interaction_contrast'].round(5))


### Exploratory interpretation — uncorrected

Averaged across five intervention sessions, the BCI group showed greater theta for distractor than no-distractor trials (0.954 vs 0.711 dB; paired difference 0.242 dB, 95% CI [0.095, 0.390], t(15) = 3.506, unadjusted p = .00319, dz = 0.876). The control contrast was −0.056 dB (0.368 vs 0.424 dB; CI [−0.220, 0.107], t(15) = −0.736, p = .473, dz = −0.184).

For distractor trials, BCI exceeded control by 0.586 dB (95% CI [0.033, 1.138], Welch t(28.50) = 2.170, unadjusted p = .0385, d = 0.767). For no-distractor trials, the group difference was 0.287 dB (CI [−0.206, 0.780], Welch t(28.81) = 1.190, p = .244, d = 0.421).

The original interaction remains a **0.299 dB larger distractor effect in BCI than control** (95% CI [0.088, 0.510]; t(30) = 2.894, t² = 8.373, p = .00703). All four exploratory p values are unadjusted; no Holm correction is applied. These session-averaged comparisons do not establish a training-induced change over sessions.


## 17. Statistical audit and exploratory session trends

Audit the apparent discrepancy between the seven-point trajectory figure and the five-session ANOVA. Compare all 320 participant/condition/intervention-session values, plotted group means/SEMs, session labels and group assignments. Independently reconstruct all seven omnibus effects using statsmodels formula-based regression, without calling the original ANOVA estimator.

Then explore linear trends over intervention sessions 1–5, separately for distractor and no-distractor trials. Each participant contributes one slope per condition. These analyses follow visual inspection and are exploratory. Use unadjusted two-sided p values, retaining the user's requested absence of GG and Holm correction. Pre/Post assessment points remain outside these five-session analyses.


In [ ]:
from posthoc_analysis.theta_intervention_audit import (
    audit_intervention_theta_statistics,
    analyze_intervention_theta_linear_trends,
)

INTERVENTION_AUDIT_DIR = INTERVENTION_ANOVA_DIR / 'statistical_audit'


### Independent ANOVA and plot audit

Regression models use participant intercepts and sum-coded factors, with separate error terms for participant means, condition values, session values, and condition-difference session values. This yields error degrees of freedom 30, 30, 120 and 120. The plotted error bars are between-participant SEM at each session; they are not confidence intervals for changes or for differences between group slopes.


In [ ]:
intervention_statistics_audit = audit_intervention_theta_statistics(
    intervention_anova_input, intervention_anova_results['anova'],
    INTERVENTION_ROOT, INTERVENTION_AUDIT_DIR,
)
display(intervention_statistics_audit['comparisons'].round(10))
display(intervention_statistics_audit['plot_values'])


### Exploratory participant-level linear trends

Fit a slope to sessions 1–5 for each participant/condition: weights [−2, −1, 0, 1, 2] / 10, in dB per session. Check each slope against an independent linear-regression calculation. Test each group's slopes against zero and compare BCI versus control slopes directly using Welch t tests. The six p values are unadjusted; 95% confidence intervals are marginal. A significant trend in one group and a nonsignificant trend in the other does not itself establish a group difference.


In [ ]:
intervention_linear_trends = analyze_intervention_theta_linear_trends(
    intervention_anova_input, INTERVENTION_AUDIT_DIR,
)
display(intervention_linear_trends['within_group'].round(5))
display(intervention_linear_trends['between_group'].round(5))


### Audit outcome and interpretation

No statistical implementation or plot-mapping error was found. All seven ANOVA effects reproduce under independent regression (maximum absolute F discrepancy < 4 × 10⁻¹⁴), and every plotted intervention mean/SEM agrees with the ANOVA input. The group × session omnibus result remains F(4,120) = 0.206, p = .935; the three-way result remains F(4,120) = 1.027, p = .396.

The distractor-trial decline in mental rehearsal is supported by the exploratory linear trend: −0.145 dB/session, 95% CI [−0.242, −0.049], t(15) = −3.220, p = .00572. BCI also has a negative estimated slope (−0.073 dB/session), but its interval is wider (95% CI [−0.230, 0.084], p = .338). This does not establish BCI stability. The direct BCI-minus-control slope difference is +0.073 dB/session, 95% CI [−0.105, 0.250], Welch t(24.89) = 0.841, p = .408.

Descriptively, distractor theta changes from 0.680 to 0.058 dB in control and 1.045 to 0.826 dB in BCI across intervention sessions 1–5. BCI participant slopes are more variable (SD 0.294 versus 0.181 dB/session). Thus the visual decline within control and nonsignificant between-group change tests are compatible. The earlier Group × Condition interaction concerns the condition contrast averaged across sessions, not its trajectory.

For no-distractor trials, slopes are −0.081 dB/session in BCI (p = .301) and −0.084 in control (p = .186); their difference is +0.003 dB/session (p = .975). No new exclusions or changes to existing analysis values were made.


## 18. Intervention response outcomes over five sessions

Plot correct, incorrect and timeout response percentages over intervention sessions 1–5 for BCI and mental rehearsal, showing pooled trials and separate distractor/no-distractor conditions. Use each participant's outcome count divided by all trials in that session/condition, then take equal-participant group means ± SEM (n = 16/group). No RT or outcome exclusions are applied; practice handling follows the validated intervention inventory.

This is a behavioral analysis, so include the 60 trials from e52 Session 3 Run 6 whose behavioral files are complete although EEG is absent. The result contains 69,060 trials across 1,151 runs; theta used the 69,000 EEG-available trials. Validate the added analysis/trigger files and cached rows explicitly.


In [ ]:
from posthoc_analysis.intervention_response_rates import (
    prepare_intervention_response_rates,
    plot_intervention_response_rates,
)
from IPython.display import Image, display

INTERVENTION_RESPONSE_DIR = INTERVENTION_ROOT / 'response_outcomes'


### Validate behavioral counts and summarize participant rates

Feedback codes: 1 = correct, 2 = incorrect, 3 = timeout. The three response types must sum to 100% within every participant/session/condition. Preserve underlying counts, denominators and run counts in exported tables. Pooled trial rates use pooled counts, with no trial duplicated within a condition.


In [ ]:
intervention_response_results = prepare_intervention_response_rates(
    INTERVENTION_CACHE, INTERVENTION_RESPONSE_DIR,
)
display(intervention_response_results['summary'])


### Pooled and condition-specific response trajectories

Columns show pooled trials, distractor and no-distractor trials; rows show correct, incorrect and timeout responses. BCI is orange and mental rehearsal blue. Points and lines are equal-participant means, error bars are SEM. All nine panels use the same 0–100% y-axis range, with ticks every 20 percentage points. No statistical tests are added.


In [ ]:
intervention_response_figure = plot_intervention_response_rates(
    intervention_response_results['participants'], intervention_response_results['summary'],
    INTERVENTION_RESPONSE_DIR,
)
display(Image(filename=intervention_response_figure['paths']['png']))


## 19. Intervention RT over five sessions with behavioral exclusions

Plot pooled, distractor and no-distractor RTs over intervention sessions 1–5 for BCI and mental rehearsal. Sequentially exclude incorrect responses, timeouts, missing/nonpositive/nonfinite RT, RT <150 ms, and values strictly outside mean ±3 sample SD among the remaining trials **within participant × session**, pooling conditions and runs. Trim once. Then exclude participants losing **more than 50%** of their trials across all five sessions; exactly 50% remains eligible.

Use all trial positions, including the first trial of each run; this plot is not conditioned on preceding feedback. Reuse GDF Status RTs from the earlier feedback analysis, joined by exact recording path and trial with stimulus/condition validation. Use the intervention inventory established in Section 14, including its documented e27 five-run exception. The missing e52 EEG run has no valid GDF-derived RT; it is a technical loss outside the participant-loss denominator. No text RT fallback is applied.


In [ ]:
from posthoc_analysis.intervention_rt_trajectory import (
    load_intervention_rt_trials,
    summarize_clean_intervention_rt,
    plot_intervention_rt_trajectory,
)
from IPython.display import Image, display

# Explicit existing repository cache; no corresponding cache exists in the study-root analyses folder.
INTERVENTION_RT_SOURCE_CACHE = REPO_ROOT / 'analyses' / 'feedback_next_trial_rt'
INTERVENTION_RT_DIR = INTERVENTION_ROOT / 'rt_trajectory'


### Load validated RTs and inspect source alignment

Verify cache hashes and source-file metadata, trial counts, unique path/trial keys, 512 Hz timing, and response boundaries. Preserve missing/ambiguous RTs as missing for explicit exclusion.


In [ ]:
intervention_rt_trials, intervention_rt_provenance = load_intervention_rt_trials(
    INTERVENTION_CACHE, INTERVENTION_RT_SOURCE_CACHE, INTERVENTION_RT_DIR,
)


### Apply trial and participant exclusions

Save the trial exclusion ledger, 160 participant/session threshold rows, participant loss report, and participant/group RT summaries. Participants are equally weighted in group means; retained trials are pooled within each participant/session/condition. Empty cells remain missing and are reported with actual sample sizes.


In [ ]:
intervention_rt_results = summarize_clean_intervention_rt(
    intervention_rt_trials, INTERVENTION_RT_DIR, max_subject_loss_percent=50,
)
display(intervention_rt_results['subjects'])
display(intervention_rt_results['summary'])


### Cleaned RT trajectories

The >50% rule excludes e29, e31, e41 and e46. BCI has 12 included participants; e26 has no eligible RT in Session 5, so BCI n = 12 in Sessions 1–4 and n = 11 in Session 5. Mental rehearsal n = 16 throughout. No values are imputed and e26 remains eligible in its other sessions. The figure labels these sample sizes.

Of 69,000 EEG-available intervention trials, 55,229 survive trial-level cleaning and 53,850 remain after participant exclusions. The three panels share a millisecond y-axis; lines show participant mean RT ± SEM, BCI orange and mental rehearsal blue. No statistical tests are added.


In [ ]:
intervention_rt_figure = plot_intervention_rt_trajectory(
    intervention_rt_results['participants'], intervention_rt_results['summary'], INTERVENTION_RT_DIR,
)
display(Image(filename=intervention_rt_figure['paths']['png']))


# Historical Pd sensitivity analyses (retained, unfiltered)

These reproduce the older unfiltered analyses requested for retention (18). They are sensitivity/historical results, distinct from the current behavior-filtered Pd analysis above. Files and figures stay in a separate historical output directory.


## Single-Run Pd Difference Wave

Compute the Pd from baseline-corrected stimulus-locked epochs using PO7/PO8 difference waves. Right-distractor trials (32) use PO7 - PO8, left-distractor trials (44) use PO8 - PO7, and no-distractor trials (8) are ignored.

In [ ]:
import sys
from pathlib import Path
import importlib

project_root = Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis')
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))

import posthoc_analysis.pd_historical as eeg
importlib.reload(eeg)

from posthoc_analysis.pd_historical import load_compute_and_plot_training_pd


In [ ]:
e21_training_run1_pd = load_compute_and_plot_training_pd(
    subject_id='e21',
    session=1,
    run=1,
)

print('Pd trial matrix shape (samples x distractor trials):', e21_training_run1_pd['pd_trials_samples_by_trials'].shape)
print('Pd trial counts:', e21_training_run1_pd['pd_code_counts'])
display(e21_training_run1_pd['pd_fig'])


## Group-Level Pd Pre/Post Waveforms

Use the saved session-condition EEG average CSV to compute Pd waveforms for each subject/session, then plot Session 1 vs Session 5 separately for the BCI and mental-rehearsal control groups. Shaded bands show SEM across subjects; the decoder construction window is shaded.


In [ ]:
from pathlib import Path
import sys
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in repo_candidates if (path / 'code' / 'src' / 'posthoc_analysis').exists()),
    Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis'),
)
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))
from posthoc_analysis.pd_historical import load_and_plot_group_pd_pre_post_from_condition_csv


In [ ]:
pd_pre_post_results = load_and_plot_group_pd_pre_post_from_condition_csv()

print('Saved figures:')
for path in pd_pre_post_results['saved_paths']:
    print(path)

print('Subject-session Pd shape:', pd_pre_post_results['subject_session_pd'].shape)
print('Group summary shape:', pd_pre_post_results['group_pd_summary'].shape)
display(pd_pre_post_results['group_pd_summary'].head())


## Pd Positive AUC Mixed ANOVA

Compute positive area under the Pd curve from 0.2 to 0.5 s for each subject in Session 1 and Session 5, then run the Group x Session mixed-design ANOVA.


In [ ]:
from pathlib import Path
import sys
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in repo_candidates if (path / 'code' / 'src' / 'posthoc_analysis').exists()),
    Path('/Users/hililbby/Library/Mobile Documents/com~apple~CloudDocs/UT Austin/JM/distractor_project/project_healthy/AI_Pd_analyzer/bci-posthoc-analysis'),
)
code_src = project_root / 'code' / 'src'
if str(code_src) not in sys.path:
    sys.path.insert(0, str(code_src))
from posthoc_analysis.pd_historical import load_compute_and_run_pd_positive_auc_anova


In [ ]:
pd_auc_anova_results = load_compute_and_run_pd_positive_auc_anova()
print('Saved AUC table:', pd_auc_anova_results['output_path'])
print('AUC table shape:', pd_auc_anova_results['pd_positive_auc'].shape)
display(pd_auc_anova_results['cell_summary'])
display(pd_auc_anova_results['anova_table'])
display(pd_auc_anova_results['planned_contrasts'])
pd_auc_anova_results['anova_diagnostics']


## Pd Negative AUC Mixed ANOVA (0.5-0.9 s)

Compute signed negative area under the Pd curve from 0.5 to 0.9 s for each subject in Session 1 and Session 5, then run the Group x Session mixed-design ANOVA. Negative AUC is defined as the integral of `min(Pd, 0)`, so more negative values indicate greater late negative Pd-area.

In [ ]:
from pathlib import Path

from posthoc_analysis.pd_historical import load_compute_and_run_pd_negative_auc_anova


In [ ]:
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path for path in repo_candidates
    if (path / 'code' / 'src' / 'posthoc_analysis').exists()
)

pd_negative_auc_results = load_compute_and_run_pd_negative_auc_anova(
    time_window=(0.5, 0.9),
    output_path=project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_negative_auc_0p5_0p9.csv',
    save_auc=True,
)

print('Saved negative AUC table:', pd_negative_auc_results['output_path'])
print('Negative AUC table shape:', pd_negative_auc_results['pd_negative_auc'].shape)


In [ ]:
display(pd_negative_auc_results['cell_summary'])
display(pd_negative_auc_results['anova_table'])
display(pd_negative_auc_results['planned_contrasts'])
pd_negative_auc_results['anova_diagnostics']


## Pd Mean Amplitude Mixed ANOVA (0.5-0.9 s)

Compute mean Pd amplitude from 0.5 to 0.9 s for each subject in Session 1 and Session 5, then run the Group x Session mixed-design ANOVA. This uses the same late post-stimulus window as the negative-AUC analysis, but evaluates signed mean amplitude rather than area.

In [ ]:
from pathlib import Path

from posthoc_analysis.pd_historical import load_compute_and_run_pd_mean_amplitude_anova


In [ ]:
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path for path in repo_candidates
    if (path / 'code' / 'src' / 'posthoc_analysis').exists()
)

pd_mean_amplitude_results = load_compute_and_run_pd_mean_amplitude_anova(
    time_window=(0.5, 0.9),
    output_path=project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_mean_amplitude_0p5_0p9.csv',
    save_table=True,
)

print('Saved mean-amplitude table:', pd_mean_amplitude_results['output_path'])
print('Mean-amplitude table shape:', pd_mean_amplitude_results['pd_mean_amplitude'].shape)


In [ ]:
display(pd_mean_amplitude_results['cell_summary'])
display(pd_mean_amplitude_results['anova_table'])
display(pd_mean_amplitude_results['planned_contrasts'])
pd_mean_amplitude_results['anova_diagnostics']


## Pd Negative AUC Mixed ANOVA (0.5-0.8 s)

Compute signed negative area under the Pd curve from 0.5 to 0.8 s for each subject in Session 1 and Session 5, then run the Group x Session mixed-design ANOVA. Negative AUC is defined as the integral of `min(Pd, 0)`, so more negative values indicate greater late negative Pd-area.

In [ ]:
from pathlib import Path

from posthoc_analysis.pd_historical import load_compute_and_run_pd_negative_auc_anova


In [ ]:
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path for path in repo_candidates
    if (path / 'code' / 'src' / 'posthoc_analysis').exists()
)

pd_negative_auc_0p5_0p8_results = load_compute_and_run_pd_negative_auc_anova(
    time_window=(0.5, 0.8),
    output_path=project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_negative_auc_0p5_0p8.csv',
    save_auc=True,
)

print('Saved negative AUC table:', pd_negative_auc_0p5_0p8_results['output_path'])
print('Negative AUC table shape:', pd_negative_auc_0p5_0p8_results['pd_negative_auc'].shape)


In [ ]:
display(pd_negative_auc_0p5_0p8_results['cell_summary'])
display(pd_negative_auc_0p5_0p8_results['anova_table'])
display(pd_negative_auc_0p5_0p8_results['planned_contrasts'])
pd_negative_auc_0p5_0p8_results['anova_diagnostics']


## Pd Negative AUC Mixed ANOVA (0.6-0.8 s)

Compute signed negative area under the Pd curve from 0.6 to 0.8 s for each subject in Session 1 and Session 5, then run the Group x Session mixed-design ANOVA. Negative AUC is defined as the integral of `min(Pd, 0)`, so more negative values indicate greater late negative Pd-area.

In [ ]:
from pathlib import Path

from posthoc_analysis.pd_historical import load_compute_and_run_pd_negative_auc_anova


In [ ]:
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path for path in repo_candidates
    if (path / 'code' / 'src' / 'posthoc_analysis').exists()
)

pd_negative_auc_0p6_0p8_results = load_compute_and_run_pd_negative_auc_anova(
    time_window=(0.6, 0.8),
    output_path=project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_negative_auc_0p6_0p8.csv',
    save_auc=True,
)

print('Saved negative AUC table:', pd_negative_auc_0p6_0p8_results['output_path'])
print('Negative AUC table shape:', pd_negative_auc_0p6_0p8_results['pd_negative_auc'].shape)


In [ ]:
display(pd_negative_auc_0p6_0p8_results['cell_summary'])
display(pd_negative_auc_0p6_0p8_results['anova_table'])
display(pd_negative_auc_0p6_0p8_results['planned_contrasts'])
pd_negative_auc_0p6_0p8_results['anova_diagnostics']


## P3/P4 Pd Negative AUC Mixed ANOVA (0.5-0.8 s)

Compute subject/session Pd waveforms using P3/P4 instead of PO7/PO8 while preserving the same contra-minus-ipsi convention: right-distractor trials use `P3 - P4`, left-distractor trials use `P4 - P3`, and no-distractor rows are validated but not included in the Pd waveform. Signed negative AUC is then computed from 0.5 to 0.8 s and analyzed with the Group x Session mixed-design ANOVA.

In [ ]:
from pathlib import Path

from posthoc_analysis.pd_historical import (
    compute_subject_session_pd_from_condition_averages,
    compute_pd_negative_auc,
    summarize_pd_negative_auc,
    run_pd_negative_auc_mixed_anova,
    run_pd_negative_auc_planned_contrasts,
)


In [ ]:
repo_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    path for path in repo_candidates
    if (path / 'code' / 'src' / 'posthoc_analysis').exists()
)

p3p4_subject_session_pd = compute_subject_session_pd_from_condition_averages(
    left_channel='P3',
    right_channel='P4',
)

p3p4_negative_auc_0p5_0p8 = compute_pd_negative_auc(
    p3p4_subject_session_pd,
    time_window=(0.5, 0.8),
)

p3p4_negative_auc_output_path = (
    project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_negative_auc_p3p4_0p5_0p8.csv'
)
p3p4_negative_auc_output_path.parent.mkdir(parents=True, exist_ok=True)
p3p4_negative_auc_0p5_0p8.to_csv(p3p4_negative_auc_output_path, index=False)
print('Saved P3/P4 negative AUC table:', p3p4_negative_auc_output_path)
print('P3/P4 negative AUC table shape:', p3p4_negative_auc_0p5_0p8.shape)


In [ ]:
p3p4_negative_auc_0p5_0p8_results = {
    'subject_session_pd': p3p4_subject_session_pd,
    'pd_negative_auc': p3p4_negative_auc_0p5_0p8,
    'cell_summary': summarize_pd_negative_auc(p3p4_negative_auc_0p5_0p8),
    'anova': run_pd_negative_auc_mixed_anova(p3p4_negative_auc_0p5_0p8),
    'planned_contrasts': run_pd_negative_auc_planned_contrasts(p3p4_negative_auc_0p5_0p8),
    'output_path': p3p4_negative_auc_output_path,
}

p3p4_negative_auc_0p5_0p8_results['anova_table'] = (
    p3p4_negative_auc_0p5_0p8_results['anova']['anova_table']
)
p3p4_negative_auc_0p5_0p8_results['anova_diagnostics'] = (
    p3p4_negative_auc_0p5_0p8_results['anova']['diagnostics']
)


In [ ]:
display(p3p4_negative_auc_0p5_0p8_results['cell_summary'])
display(p3p4_negative_auc_0p5_0p8_results['anova_table'])
display(p3p4_negative_auc_0p5_0p8_results['planned_contrasts'])
p3p4_negative_auc_0p5_0p8_results['anova_diagnostics']


## Publication-ready Pd pre/post waveform figure

Create a two-panel PO7/PO8 contralateral-minus-ipsilateral Pd figure. The panels show BCI and mental-rehearsal groups with matched axes, pre-training/post-training mean waveforms, SEM bands, group sample sizes, the prespecified 0.2–0.5 s positive-Pd AUC window, and color-matched dotted lines for the mean distractor-trial reaction time in each group and timepoint. The shared x-axis extends through the latest mean reaction time.

In [ ]:
from posthoc_analysis.pd_historical import (
    load_and_plot_combined_group_pd_pre_post_from_condition_csv,
)


In [ ]:
pd_publication_figure_results = (
    load_and_plot_combined_group_pd_pre_post_from_condition_csv(
        output_dir=project_root / 'figures' / 'historical_pd',
        save_figure=True,
    )
)
pd_publication_figure_results['saved_paths']


## Right- and left-distractor Pd waveform figures

Create matched two-panel pre-training/post-training Pd figures for each distractor side. Right-distractor Pd is calculated as PO7 − PO8; left-distractor Pd is calculated as PO8 − PO7. The figures retain matched BCI/control axes, SEM bands, the prespecified AUC window, and mean distractor-trial RT reference lines.

In [ ]:
from posthoc_analysis.pd_historical import (
    load_and_plot_lateralized_pd_pre_post_from_condition_csv,
)


In [ ]:
lateralized_pd_figure_results = (
    load_and_plot_lateralized_pd_pre_post_from_condition_csv(
        output_dir=project_root / 'figures' / 'historical_pd',
        save_figures=True,
    )
)
{
    side: result['saved_paths']
    for side, result in lateralized_pd_figure_results.items()
}


## Right- and left-distractor positive Pd AUC statistics

Run the prespecified 0.2–0.5 s positive-Pd AUC Group × Time mixed ANOVA separately for right- and left-distractor waveforms. Right-distractor Pd is PO7 − PO8; left-distractor Pd is PO8 − PO7.

In [ ]:
from posthoc_analysis.pd_historical import load_compute_and_run_pd_positive_auc_anova


In [ ]:
right_distractor_pd_positive_auc_results = (
    load_compute_and_run_pd_positive_auc_anova(
        distractor_condition='distractor_right',
        output_path=(
            project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_positive_auc_0p2_0p5_right_distractor.csv'
        ),
    )
)
left_distractor_pd_positive_auc_results = (
    load_compute_and_run_pd_positive_auc_anova(
        distractor_condition='distractor_left',
        output_path=(
            project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_positive_auc_0p2_0p5_left_distractor.csv'
        ),
    )
)


In [ ]:
for side_label, result in {
    'Right distractor (PO7 − PO8)': right_distractor_pd_positive_auc_results,
    'Left distractor (PO8 − PO7)': left_distractor_pd_positive_auc_results,
}.items():
    print(f'\n{side_label}')
    display(result['cell_summary'])
    display(result['anova_table'])
    display(result['planned_contrasts'])


## Combined Pd mean absolute amplitude statistics (0.2–0.5 s)

For the balanced, combined left- and right-distractor Pd waveform, compute each participant's mean absolute Pd amplitude across the prespecified 200–500 ms window. This quantifies waveform magnitude regardless of polarity; it is therefore distinct from positive-Pd AUC. The Group × Session mixed ANOVA and planned pre/post contrasts are exploratory unless this measure was preregistered.

In [ ]:
from posthoc_analysis.pd_historical import load_compute_and_run_pd_mean_absolute_amplitude_anova


In [ ]:
combined_pd_mean_absolute_results = (
    load_compute_and_run_pd_mean_absolute_amplitude_anova(
        output_path=(
            project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_mean_absolute_amplitude_0p2_0p5.csv'
        ),
    )
)


In [ ]:
display(combined_pd_mean_absolute_results['cell_summary'])
display(combined_pd_mean_absolute_results['anova_table'])
display(combined_pd_mean_absolute_results['planned_contrasts'])


## Combined Pd pre/post cluster-permutation statistics

Test the balanced left/right Pd waveform separately in the BCI and mental-rehearsal groups with paired, two-sided cluster-based permutation tests (post minus pre). The requested 0–1200 ms range is tested as far as the available stimulus-locked data permit (0–998 ms). Cluster p-values control the family-wise error rate across time samples within each group.

In [ ]:
from posthoc_analysis.pd_historical import load_run_and_plot_pd_pre_post_cluster_permutation


In [ ]:
pd_cluster_permutation_results = load_run_and_plot_pd_pre_post_cluster_permutation(
    output_dir=project_root / 'figures' / 'historical_pd',
    n_permutations=10_000,
    random_seed=20260811,
)


In [ ]:
display(pd_cluster_permutation_results['cluster_results']['group_summary'])
display(pd_cluster_permutation_results['cluster_results']['clusters'])
pd_cluster_permutation_results['figure_paths']


## Combined Pd late negative AUC statistics (600–998 ms)

Compute the signed negative area of the balanced left/right Pd waveform from 600 ms through the final available sample (998.047 ms). At each time point, positive values are set to zero before trapezoidal integration; more-negative AUC values therefore indicate a larger negative-going waveform area. Test Group × Session effects and planned within-group pre/post changes.

In [ ]:
from posthoc_analysis.pd_historical import load_compute_and_run_pd_negative_auc_anova


In [ ]:
late_negative_auc_results = load_compute_and_run_pd_negative_auc_anova(
    time_window=(0.6, 0.998046875),
    output_path=(
        project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_negative_auc_0p6_0p998.csv'
    ),
)


In [ ]:
display(late_negative_auc_results['cell_summary'])
display(late_negative_auc_results['anova_table'])
display(late_negative_auc_results['planned_contrasts'])


## Combined Pd late mean-amplitude statistics (600–998 ms)

Using the same balanced left/right Pd waveform and late 600–998.047 ms window, compute each participant's signed mean Pd amplitude across samples. Unlike negative AUC, this measure retains both positive and negative values and has units of µV.

In [ ]:
from posthoc_analysis.pd_historical import load_compute_and_run_pd_mean_amplitude_anova


In [ ]:
late_mean_amplitude_results = load_compute_and_run_pd_mean_amplitude_anova(
    time_window=(0.6, 0.998046875),
    output_path=(
        project_root / 'exploratory_results' / 'historical_pd' / 'training_pd_mean_amplitude_0p6_0p998.csv'
    ),
)


In [ ]:
display(late_mean_amplitude_results['cell_summary'])
display(late_mean_amplitude_results['anova_table'])
display(late_mean_amplitude_results['planned_contrasts'])


# Pd robustness and response-timing controls

Retained analysis 19 and target-geometry controls from 15. The raw extraction cache is shared historical evidence; alpha/theta candidate inference is excluded. Pd/late-voltage reliability and controls remain exploratory. The saved historical control RT table remains explicit rather than replacing the primary corrected behavioral analysis.

Saved historical visual records: [Pd robustness](../exploratory_results/reports/09_pd_robustness_and_selection.pdf), [late response controls](../exploratory_results/reports/06_late_response_controls.pdf), and [target geometry](../exploratory_results/figures/behavior_geometry_controls.pdf). These dated reports describe the original analysis; the current filtered-Pd results above remain primary.


In [ ]:
from posthoc_analysis.pd_robustness import (
    load_neural_checkpoints, response_and_count_controls,
    reconcile_pd_and_reliability, behavioral_geometry_controls,
)
from posthoc_analysis.pd_robustness_support import compare_group_changes

pd_robustness_root = project_root / 'exploratory_results'
pd_robustness_output = pd_robustness_root / 'tables' / 'pd_robustness'
pd_robustness_output.mkdir(parents=True, exist_ok=True)


In [ ]:
pd_robustness_trials, pd_robustness_arrays, pd_robustness_qc = load_neural_checkpoints(
    pd_robustness_root / 'neural_response_controls',
    project_root / 'neural_exploration_settings.json',
)
display(pd_robustness_qc)


In [ ]:
pd_control_summary, pd_control_counts, pd_control_waves, pd_matched_trials = response_and_count_controls(
    pd_robustness_trials, pd_robustness_arrays,
)
pd_control_tests, pd_control_changes = compare_group_changes(
    pd_control_summary, ['late_all_correct', 'late_matched', 'response_pre_matched', 'response_post_matched', 'pd_auc_equal_count'],
)
display(pd_control_tests)
display(pd_control_counts)


In [ ]:
pd_reliability = reconcile_pd_and_reliability(
    pd_robustness_trials, pd_robustness_arrays, pd_robustness_output, n_splits=200, seed=20261015,
)
display(pd_reliability['pd_threshold_sensitivity'])
display(pd_reliability['reliability_summary'])


In [ ]:
pd_geometry = behavioral_geometry_controls(
    pd_robustness_trials,
    pd_robustness_root / 'tables' / 'corrected_rt_statistics' / 'trial_data_clean.csv',
    pd_robustness_output,
)
display(pd_geometry['tests'])
